# MAVIS — Multimer-Aware Variant Impact Scoring
## Full Analysis Pipeline

Complete structural + annotation concordance pipeline. Includes four-way concordance (structure × ΔΔG × AlphaMissense × Franklin/ClinVar), dual scoring pipelines, and pipeline agreement tracking.

**Inputs:** Variant CSV + AlphaFold structures + (optional) AlphaMissense/Franklin annotations

**Outputs:** Structural tiers, mechanisms, concordance scores, multi-sheet XLSX

See [README.md](README.md) for full documentation.

## Cell 1: Configuration

Edit paths and structure definitions.

In [ ]:
# =============================================================================
# CELL 1: CONFIGURATION
# =============================================================================

import os, sys, re, warnings, subprocess, tempfile
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Set
import pandas as pd
import numpy as np
from Bio.PDB import PDBParser, MMCIFParser, NeighborSearch, ShrakeRupley

warnings.filterwarnings('ignore')

# =============================================================================
# PATHS — adjust these to your local environment
# =============================================================================
WORKING_DIR    = Path(".")
RESULTS_DIR    = WORKING_DIR / "results"
DSSP_PATH      = "mkdssp"

for d in [RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Directories to search for structure files (in priority order)
SEARCH_DIRS = [
    WORKING_DIR,
    WORKING_DIR / "structures",
    WORKING_DIR / "structures" / "monomers",
    WORKING_DIR / "structures" / "multimers",
]

def find_file(filename):
    """Search multiple directories for a file."""
    if filename is None:
        return None
    for d in SEARCH_DIRS:
        p = d / filename
        if p.exists():
            return p
    return None

# =============================================================================
# INPUT FILES
# =============================================================================
VARIANTS_FILE    = WORKING_DIR / "variants.csv"
MONOMER_DDG_FILE = WORKING_DIR / "foldx_ddg_monomer_results_all.csv"
MULTIMER_DDG_FILE = WORKING_DIR / "foldx_ddg_multimer_results.csv"

# =============================================================================
# MONOMER STRUCTURE DEFINITIONS
# gene → (cif_filename_or_None, pdb_filename_or_None)
# CIF preferred for pLDDT on shroom3/cdh2/dvl2/ctnnb1/zic3/actb (PDB B-factors=0)
# PDB B-factors valid for gli3/kpna1/kpna6/mdfi/rock2/tcf7l1
# =============================================================================
# =============================================================================
# STRUCTURE DEFINITIONS
# =============================================================================
# Option A: Set AUTO_DETECT = True (below) and place structures in STRUCTURE_DIR
# Option B: Set AUTO_DETECT = False and manually define these dictionaries
#
# MONOMER: gene -> (cif_file_or_None, pdb_file_or_None)
# MULTIMER: (gene1, partner_label, cif, pdb, chain1, chain2, is_primary)
MONOMER_STRUCTURES = {}
# =============================================================================
# MULTIMER COMPLEX DEFINITIONS
# (gene1, partner_label, cif_file_or_None, pdb_file, chain_gene1, chain_partner, is_primary)
# NOTE: For pLDDT, PDB B-factors are used (not CIF) because CIF chain order
# varies by complex (e.g., actb CIF has chains swapped vs PDB).
# =============================================================================
MULTIMER_STRUCTURES = [
    ('shroom3','actin',         'fold_shroom3_actin_chain_model_0.cif',            'fold_shroom3_actin_chain_model_0.pdb',        'A','B', True),
    ('shroom3','actb',          'fold_shroom3_actb_model_0.cif',                   'fold_shroom3_actb_model_0.pdb',               'B','A', False),
    ('shroom3','dvl2',          'fold_shroom3_dvl2_model_0.cif',                   'fold_shroom3_dvl2_model_0.pdb',               'A','B', True),
    ('shroom3','cdh2_truncated','fold_shroom3_cdh2_truncated_model_0.cif',         'fold_shroom3_cdh2_truncated_model_0.pdb',     'A','B', False),
    ('shroom3','ctnnb1',        'fold_shroom3_ctnnb1_model_0.cif',                 'fold_shroom3_ctnnb1_model_0.pdb',             'A','B', True),
    ('shroom3','cdh2_cyto',     'fold_shroom3_cdh2_cytoplasmic_domain_model_0.cif','fold_shroom3_cdh2_cytoplasmic_domain.pdb',    'A','B', True),
    ('shroom3','actb_no_bind',  'fold_shroom3_no_actin_binding_actb_chain_model_0.cif','fold_shroom3_no_actin_binding_actb_chain.pdb','A','B', False),
    ('shroom3','rock2',         None,                                              'fold_shroom3_rock2_model_0.pdb',              'A','B', True),
    ('zic3','gli3',   'fold_zic3_gli3_model_0.cif',   'fold_zic3_gli3_model_0.pdb',  'A','B', True),
    ('zic3','kpna1',  'fold_zic3_kpna1_model_0.cif',  'fold_zic3_kpna1_model_0.pdb', 'A','B', True),
    ('zic3','kpna6',  'fold_zic3_kpna6_model_0.cif',  'fold_zic3_kpna6_model_0.pdb', 'A','B', True),
    ('zic3','mdfi',   'fold_zic3_mdfi_model_0.cif',   'fold_zic3_mdfi_model_0.pdb',  'A','B', True),
    ('zic3','tcf7l1', 'fold_zic3_tcf7l1_model_0.cif', 'fold_zic3_tcf7l1_model_0.pdb','A','B', True),
]

# =============================================================================
# AMINO ACID DATA
# =============================================================================
THREE_TO_ONE = {
    'ALA':'A','CYS':'C','ASP':'D','GLU':'E','PHE':'F','GLY':'G','HIS':'H',
    'ILE':'I','LYS':'K','LEU':'L','MET':'M','ASN':'N','PRO':'P','GLN':'Q',
    'ARG':'R','SER':'S','THR':'T','VAL':'V','TRP':'W','TYR':'Y',
}
AA_PROPERTIES = {
    'A':{'size':'small','charge':'neutral','hydrophobic':True},
    'R':{'size':'large','charge':'positive','hydrophobic':False},
    'N':{'size':'medium','charge':'neutral','hydrophobic':False},
    'D':{'size':'medium','charge':'negative','hydrophobic':False},
    'C':{'size':'small','charge':'neutral','hydrophobic':True},
    'E':{'size':'medium','charge':'negative','hydrophobic':False},
    'Q':{'size':'medium','charge':'neutral','hydrophobic':False},
    'G':{'size':'small','charge':'neutral','hydrophobic':False},
    'H':{'size':'medium','charge':'positive','hydrophobic':False},
    'I':{'size':'medium','charge':'neutral','hydrophobic':True},
    'L':{'size':'medium','charge':'neutral','hydrophobic':True},
    'K':{'size':'large','charge':'positive','hydrophobic':False},
    'M':{'size':'medium','charge':'neutral','hydrophobic':True},
    'F':{'size':'large','charge':'neutral','hydrophobic':True},
    'P':{'size':'small','charge':'neutral','hydrophobic':False},
    'S':{'size':'small','charge':'neutral','hydrophobic':False},
    'T':{'size':'small','charge':'neutral','hydrophobic':False},
    'W':{'size':'large','charge':'neutral','hydrophobic':True},
    'Y':{'size':'large','charge':'neutral','hydrophobic':False},
    'V':{'size':'small','charge':'neutral','hydrophobic':True},
}
# Max SASA (Tien et al 2013, theoretical Gly-X-Gly)
MAX_SASA = {
    'A':129,'R':274,'N':195,'D':193,'C':167,'E':223,'Q':225,'G':104,
    'H':224,'I':197,'L':201,'K':236,'M':224,'F':240,'P':159,'S':155,
    'T':172,'V':174,'W':285,'Y':263,
}

GRANTHAM = {
    ('A','R'):112,('A','N'):111,('A','D'):126,('A','C'):195,('A','Q'):91,('A','E'):107,
    ('A','G'):60,('A','H'):86,('A','I'):94,('A','L'):96,('A','K'):106,('A','M'):84,
    ('A','F'):113,('A','P'):27,('A','S'):99,('A','T'):58,('A','W'):148,('A','Y'):112,('A','V'):64,
    ('R','N'):86,('R','D'):96,('R','C'):180,('R','Q'):43,('R','E'):54,('R','G'):125,
    ('R','H'):29,('R','I'):97,('R','L'):102,('R','K'):26,('R','M'):91,('R','F'):97,
    ('R','P'):103,('R','S'):110,('R','T'):71,('R','W'):101,('R','Y'):77,('R','V'):96,
    ('N','D'):23,('N','C'):139,('N','Q'):46,('N','E'):42,('N','G'):80,('N','H'):68,
    ('N','I'):149,('N','L'):153,('N','K'):94,('N','M'):142,('N','F'):158,('N','P'):91,
    ('N','S'):46,('N','T'):65,('N','W'):174,('N','Y'):143,('N','V'):133,
    ('D','C'):154,('D','Q'):61,('D','E'):45,('D','G'):94,('D','H'):81,('D','I'):168,
    ('D','L'):172,('D','K'):101,('D','M'):160,('D','F'):177,('D','P'):108,('D','S'):65,
    ('D','T'):85,('D','W'):181,('D','Y'):160,('D','V'):152,
    ('C','Q'):154,('C','E'):170,('C','G'):159,('C','H'):174,('C','I'):198,('C','L'):198,
    ('C','K'):202,('C','M'):196,('C','F'):205,('C','P'):169,('C','S'):112,('C','T'):149,
    ('C','W'):215,('C','Y'):194,('C','V'):192,
    ('Q','E'):29,('Q','G'):87,('Q','H'):24,('Q','I'):109,('Q','L'):113,('Q','K'):53,
    ('Q','M'):101,('Q','F'):116,('Q','P'):76,('Q','S'):68,('Q','T'):42,('Q','W'):130,
    ('Q','Y'):99,('Q','V'):96,
    ('E','G'):98,('E','H'):40,('E','I'):134,('E','L'):138,('E','K'):56,('E','M'):126,
    ('E','F'):140,('E','P'):93,('E','S'):80,('E','T'):65,('E','W'):152,('E','Y'):122,('E','V'):121,
    ('G','H'):98,('G','I'):135,('G','L'):138,('G','K'):127,('G','M'):127,('G','F'):153,
    ('G','P'):42,('G','S'):56,('G','T'):59,('G','W'):184,('G','Y'):147,('G','V'):109,
    ('H','I'):94,('H','L'):99,('H','K'):32,('H','M'):87,('H','F'):100,('H','P'):77,
    ('H','S'):89,('H','T'):47,('H','W'):115,('H','Y'):83,('H','V'):84,
    ('I','L'):5,('I','K'):102,('I','M'):10,('I','F'):21,('I','P'):95,('I','S'):142,
    ('I','T'):89,('I','W'):61,('I','Y'):33,('I','V'):29,
    ('L','K'):107,('L','M'):15,('L','F'):22,('L','P'):98,('L','S'):145,('L','T'):92,
    ('L','W'):61,('L','Y'):36,('L','V'):32,
    ('K','M'):95,('K','F'):102,('K','P'):103,('K','S'):121,('K','T'):78,('K','W'):110,
    ('K','Y'):85,('K','V'):97,
    ('M','F'):28,('M','P'):87,('M','S'):135,('M','T'):81,('M','W'):67,('M','Y'):36,('M','V'):21,
    ('F','P'):114,('F','S'):155,('F','T'):103,('F','W'):40,('F','Y'):22,('F','V'):50,
    ('P','S'):74,('P','T'):38,('P','W'):147,('P','Y'):110,('P','V'):68,
    ('S','T'):58,('S','W'):177,('S','Y'):144,('S','V'):124,
    ('T','W'):128,('T','Y'):92,('T','V'):69,
    ('W','Y'):37,('W','V'):88,
    ('Y','V'):55,
}

def get_grantham(a1, a2):
    if pd.isna(a1) or pd.isna(a2): return -1
    a1, a2 = str(a1).upper(), str(a2).upper()
    if a1 == a2: return 0
    return GRANTHAM.get((a1,a2), GRANTHAM.get((a2,a1), -1))

def classify_grantham(d):
    if pd.isna(d) or d is None or d < 0: return 'unknown'
    d = int(d)
    if d <= 50: return 'conservative'
    elif d <= 100: return 'moderately_conservative'
    elif d <= 150: return 'moderately_radical'
    else: return 'radical'

def grantham_severity(d):
    if pd.isna(d) or d is None or d < 0: return 0.0
    return min(4.0, float(d) / 53.75)

def get_property_changes(r, a):
    if pd.isna(r) or pd.isna(a): return 'unknown'
    p1, p2 = AA_PROPERTIES.get(str(r).upper(),{}), AA_PROPERTIES.get(str(a).upper(),{})
    if not p1 or not p2: return 'unknown'
    ch = []
    for k in ['size','charge','hydrophobic']:
        if p1.get(k) != p2.get(k): ch.append(f"{k}:{p1[k]}->{p2[k]}")
    return ';'.join(ch) if ch else 'none'

def sf(v, d=0.0):
    if pd.isna(v) or v is None: return d
    try: return float(v)
    except: return d
def si(v, d=0):
    if pd.isna(v) or v is None: return d
    try: return int(v)
    except: return d
def ss(v): return '' if pd.isna(v) or v is None else str(v)
def sb(v, d=False):
    if pd.isna(v) or v is None: return d
    return bool(v)

# Verify files exist
found = 0
for g, (cif, pdb) in MONOMER_STRUCTURES.items():
    cf = find_file(cif)
    pf = find_file(pdb)
    if cf or pf:
        found += 1
    else:
        print(f"  ⚠ {g}: no structure found (tried {cif}, {pdb})")
print(f"✓ Configuration loaded: {found}/{len(MONOMER_STRUCTURES)} monomer structures found")
print(f"  Multimer complexes: {len(MULTIMER_STRUCTURES)}")

mfound = 0
for g1, pl, cif, pdb, *_ in MULTIMER_STRUCTURES:
    pf = find_file(pdb)
    if pf: mfound += 1
    else: print(f"  ⚠ {g1}-{pl}: {pdb} NOT FOUND")
print(f"  Multimer PDBs found: {mfound}/{len(MULTIMER_STRUCTURES)}")

# =============================================================================
# DRY RUN MODE
# =============================================================================
# When DRY_RUN = True, FoldX cells are skipped and mock DDG values are used.
# Useful for testing pipeline logic without a FoldX installation.
DRY_RUN = False


## Cell 2: Structure Loading Functions

In [ ]:
# =============================================================================
# CELL 2: STRUCTURE LOADING AND EXTRACTION FUNCTIONS
# =============================================================================

_pdb_parser = PDBParser(QUIET=True)
_cif_parser = MMCIFParser(QUIET=True)


def load_cif(path):
    if path and path.exists():
        try: return _cif_parser.get_structure('s', str(path))
        except: pass
    return None

def load_pdb(path):
    if path and path.exists():
        try: return _pdb_parser.get_structure('s', str(path))
        except: pass
    return None


def get_plddt(structure, chain_id='A'):
    """Per-residue pLDDT from B-factors. Returns only non-zero values."""
    plddt = {}
    if structure is None: return plddt
    model = structure[0]
    # Resolve chain
    if chain_id not in model:
        for c in model: chain_id = c.id; break
    if chain_id not in model: return plddt
    for res in model[chain_id].get_residues():
        if res.id[0] == ' ':
            p = None
            if 'CA' in res: p = res['CA'].bfactor
            else:
                for atom in res: p = atom.bfactor; break
            if p is not None and p > 0:
                plddt[res.id[1]] = round(p, 2)
    return plddt


def get_monomer_plddt(gene_lower):
    """Get pLDDT for monomer: CIF first (needed for shroom3/cdh2/dvl2/ctnnb1/zic3/actb),
    then PDB fallback."""
    cif_name, pdb_name = MONOMER_STRUCTURES.get(gene_lower, (None, None))
    cif_path = find_file(cif_name)
    pdb_path = find_file(pdb_name)

    # Try CIF first
    if cif_path:
        struct = load_cif(cif_path)
        if struct:
            plddt = get_plddt(struct, 'A')
            if plddt:
                return plddt, struct, 'cif', cif_path
    # PDB fallback
    if pdb_path:
        struct = load_pdb(pdb_path)
        if struct:
            plddt = get_plddt(struct, 'A')
            if plddt:
                return plddt, struct, 'pdb', pdb_path
            # Even if pLDDT empty, return struct for contacts
            return {}, struct, 'pdb_no_plddt', pdb_path
    return {}, None, None, None


def get_multimer_plddt(pdb_path, cif_path, chain_id):
    """Get pLDDT for multimer: PDB first (consistent chain order), CIF fallback.
    AlphaFold multimer PDBs should have valid B-factors."""
    # PDB first — chain order is consistent
    struct = load_pdb(pdb_path)
    if struct:
        plddt = get_plddt(struct, chain_id)
        if plddt:
            return plddt, 'pdb'
    # CIF fallback (WARNING: chain order may differ!)
    struct = load_cif(cif_path)
    if struct:
        plddt = get_plddt(struct, chain_id)
        if plddt:
            return plddt, 'cif'
    return {}, None


def get_residue_aa(structure, chain_id='A'):
    if structure is None: return {}
    model = structure[0]
    if chain_id not in model:
        for c in model: chain_id = c.id; break
    if chain_id not in model: return {}
    return {r.id[1]: THREE_TO_ONE.get(r.resname, '?')
            for r in model[chain_id].get_residues() if r.id[0] == ' '}


def count_contacts(structure, chain_id='A', distance=5.0):
    """Unique residue-residue contacts, sequence separation >= 3.
    Matches original working pipeline (cell 24 of 85-cell notebook)."""
    if structure is None: return {}
    model = structure[0]
    if chain_id not in model:
        for c in model: chain_id = c.id; break
    if chain_id not in model: return {}

    chain = model[chain_id]
    residues = [r for r in chain.get_residues() if r.id[0] == ' ']
    contacts = {}

    for i, res in enumerate(residues):
        pos = res.id[1]
        neighbor_set = set()
        for j, other in enumerate(residues):
            if abs(i - j) < 3:  # skip self and immediate neighbors
                continue
            for atom_i in res.get_atoms():
                found = False
                for atom_j in other.get_atoms():
                    if atom_i - atom_j < distance:
                        neighbor_set.add(other.id[1])
                        found = True
                        break
                if found:
                    break
        contacts[pos] = len(neighbor_set)
    return contacts


def count_interface(structure, my_chain, partner_chain, distance=5.0):
    """Inter-chain contacts: unique partner residues within distance for each residue."""
    if structure is None: return {}, set()
    model = structure[0]
    if my_chain not in model or partner_chain not in model:
        return {}, set()

    partner_atoms = list(model[partner_chain].get_atoms())
    if not partner_atoms: return {}, set()
    ns = NeighborSearch(partner_atoms)

    inter, iface = {}, set()
    for res in model[my_chain].get_residues():
        if res.id[0] != ' ': continue
        partner_residues = set()
        for atom in res.get_atoms():
            for nb in ns.search(atom.coord, distance, 'R'):
                if nb.id[0] == ' ':
                    partner_residues.add(nb.id[1])
        count = len(partner_residues)
        if count > 0:
            inter[res.id[1]] = count
            iface.add(res.id[1])
    return inter, iface


def get_accessibility(structure, chain_id='A'):
    """Relative solvent accessibility using ShrakeRupley (no external tools).
    Compatible with Biopython 1.86."""
    acc = {}
    if structure is None: return acc
    model = structure[0]
    if chain_id not in model:
        for c in model: chain_id = c.id; break
    if chain_id not in model: return acc
    try:
        sr = ShrakeRupley()
        sr.compute(structure[0], level='R')
        for res in model[chain_id].get_residues():
            if res.id[0] != ' ': continue
            aa = THREE_TO_ONE.get(res.resname, 'X')
            max_s = MAX_SASA.get(aa, 200)
            rel = min(1.0, res.sasa / max_s) if max_s > 0 and hasattr(res, 'sasa') else None
            if rel is not None:
                acc[res.id[1]] = round(rel, 9)
    except Exception as e:
        print(f"    ShrakeRupley warning: {e}")
    return acc


def get_secondary_structure(pdb_path, chain_id='A'):
    """Secondary structure via mkdssp subprocess (handles v4 output format).
    Falls back to empty if mkdssp fails."""
    ss_map = {}
    if pdb_path is None or not pdb_path.exists(): return ss_map
    try:
        # Try mkdssp v4 with classic DSSP output format
        for cmd in [
            [DSSP_PATH, '--output-format', 'dssp', str(pdb_path)],
            [DSSP_PATH, '-i', str(pdb_path)],
            [DSSP_PATH, str(pdb_path)],
        ]:
            try:
                result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
                if result.returncode == 0 and len(result.stdout) > 100:
                    break
            except FileNotFoundError:
                return ss_map
        else:
            return ss_map

        if result.returncode != 0:
            return ss_map

        # Parse DSSP output
        in_data = False
        for line in result.stdout.split('\n'):
            if '  #  RESIDUE' in line:
                in_data = True
                continue
            if not in_data or len(line) < 17:
                continue
            # Skip chain break lines
            if line[13] == '!':
                continue
            try:
                chain = line[11]
                if chain != chain_id:
                    continue
                resnum_str = line[5:10].strip()
                if not resnum_str:
                    continue
                resnum = int(resnum_str)
                sec = line[16] if len(line) > 16 and line[16] != ' ' else '-'
                ss_map[resnum] = sec
            except (ValueError, IndexError):
                continue
    except Exception:
        pass
    return ss_map


def classify_burial(a):
    if a is None or pd.isna(a): return 'unknown'
    if float(a) < 0.05: return 'buried_core'
    elif float(a) < 0.25: return 'partially_buried'
    else: return 'surface_exposed'

def classify_plddt(v):
    if v is None or pd.isna(v): return 'unknown'
    if v >= 90: return 'very_high'
    elif v >= 70: return 'confident'
    elif v >= 50: return 'low'
    else: return 'very_low'

def classify_contacts(n):
    if n is None or pd.isna(n): return 'unknown'
    if n >= 8: return 'high_contact'
    elif n >= 1: return 'medium_contact'
    else: return 'low_contact'

print("✓ Extraction functions defined")

# =============================================================================
# STRUCTURE AUTO-DETECTION
# =============================================================================
# When AUTO_DETECT = True, MAVIS scans STRUCTURE_DIR for PDB/CIF files and
# automatically populates MONOMER_STRUCTURES and MULTIMER_STRUCTURES.
#
# Naming conventions recognized:
#   Monomer: fold_{gene}_model_N.pdb, {gene}.pdb, {gene}_model_N.pdb
#   Multimer: fold_{gene1}_{gene2}_model_N.pdb, {gene1}_{gene2}.pdb
#
# Auto-detection uses chain count to distinguish monomer (1 chain) from
# multimer (2+ chains). Override with manual config for complex cases.
# =============================================================================

STRUCTURE_DIR = WORKING_DIR / "structures"
AUTO_DETECT = True   # Set False to use manual MONOMER_STRUCTURES / MULTIMER_STRUCTURES

def auto_detect_structures(structure_dir):
    """Scan directory for AlphaFold structures and classify as monomer/multimer."""
    structure_dir = Path(structure_dir)
    if not structure_dir.exists():
        print(f"  ⚠ Structure directory not found: {structure_dir}")
        return {}, []
    
    pdb_files = sorted(structure_dir.glob("*.pdb"))
    cif_files = sorted(structure_dir.glob("*.cif"))
    all_files = pdb_files + cif_files
    
    if not all_files:
        print(f"  ⚠ No PDB/CIF files found in {structure_dir}")
        return {}, []
    
    print(f"  Found {len(pdb_files)} PDB + {len(cif_files)} CIF files in {structure_dir}")
    
    mono = {}   # gene -> (cif_path, pdb_path)
    multi = []  # list of tuples
    
    # Group files by base name (without extension and model suffix)
    import re
    file_groups = {}  # base_key -> {'pdb': path, 'cif': path}
    
    for f in all_files:
        name = f.stem.lower()
        ext = f.suffix.lower()
        # Strip model suffix: _model_0, _model_1, etc.
        base = re.sub(r'_model_\d+$', '', name)
        if base not in file_groups:
            file_groups[base] = {}
        file_groups[base][ext.replace('.', '')] = f
    
    for base, files in sorted(file_groups.items()):
        pdb_path = files.get('pdb')
        cif_path = files.get('cif')
        
        # Determine chain count from PDB (or CIF)
        struct_path = pdb_path or cif_path
        n_chains = 0
        chain_ids = []
        try:
            if str(struct_path).endswith('.pdb'):
                s = _pdb_parser.get_structure('s', str(struct_path))
            else:
                s = _cif_parser.get_structure('s', str(struct_path))
            model = s[0]
            chain_ids = [c.id for c in model.get_chains()]
            n_chains = len(chain_ids)
        except Exception as e:
            print(f"    ⚠ Could not parse {struct_path.name}: {e}")
            continue
        
        # Parse gene name(s) from filename
        # Remove 'fold_' prefix
        clean = re.sub(r'^fold_', '', base)
        
        if n_chains == 1:
            # Monomer
            gene = clean.split('_')[0] if '_' in clean else clean
            cif_name = cif_path.name if cif_path else None
            pdb_name = pdb_path.name if pdb_path else None
            if gene not in mono:
                mono[gene] = (cif_name, pdb_name)
                print(f"    Monomer: {gene} ({n_chains} chain) — {struct_path.name}")
            else:
                # Update with any new file format
                existing_cif, existing_pdb = mono[gene]
                mono[gene] = (cif_name or existing_cif, pdb_name or existing_pdb)
        
        elif n_chains >= 2:
            # Multimer — parse gene1 and partner from filename
            parts = clean.split('_')
            if len(parts) >= 2:
                gene1 = parts[0]
                # Partner is everything between gene1 and 'model' (or end)
                partner_parts = []
                for p in parts[1:]:
                    if p == 'model' or p.isdigit():
                        break
                    partner_parts.append(p)
                partner = '_'.join(partner_parts) if partner_parts else parts[1]
            else:
                gene1 = clean
                partner = f"partner_{len(multi)}"
            
            cif_name = cif_path.name if cif_path else None
            pdb_name = pdb_path.name if pdb_path else None
            
            if pdb_name or cif_name:
                multi.append((
                    gene1, partner, cif_name, pdb_name or cif_name,
                    chain_ids[0], chain_ids[1] if len(chain_ids) > 1 else 'B', True
                ))
                print(f"    Multimer: {gene1}—{partner} ({n_chains} chains: {chain_ids}) — {struct_path.name}")
    
    return mono, multi


if AUTO_DETECT:
    print("\n=== Auto-detecting structures ===")
    _auto_mono, _auto_multi = auto_detect_structures(STRUCTURE_DIR)
    if not MONOMER_STRUCTURES:
        MONOMER_STRUCTURES.update(_auto_mono)
    if not MULTIMER_STRUCTURES:
        MULTIMER_STRUCTURES.extend(_auto_multi)
    print(f"\n  Result: {len(MONOMER_STRUCTURES)} monomers, {len(MULTIMER_STRUCTURES)} multimers")


## Cell 3: Load Variants

In [ ]:
# =============================================================================
# CELL 3: LOAD VARIANTS
# =============================================================================

# Support both ref_aa/alt_aa and combined variant column formats
variants_df = pd.read_csv(VARIANTS_FILE, encoding='utf-8-sig')
variants_df.columns = [c.lower().strip() for c in variants_df.columns]

# Auto-parse combined variant column if present
if 'variant' in variants_df.columns and 'ref_aa' not in variants_df.columns:
    import re as _re
    def _parse_variant(v):
        m = _re.match(r'([A-Z])(\d+)([A-Z])', str(v).upper())
        if m:
            return m.group(1), int(m.group(2)), m.group(3)
        return None, None, None
    
    parsed = variants_df['variant'].apply(_parse_variant)
    variants_df['ref_aa'] = [p[0] for p in parsed]
    variants_df['position'] = [p[1] for p in parsed]
    variants_df['alt_aa'] = [p[2] for p in parsed]
    variants_df = variants_df.dropna(subset=['position'])
    print(f"  Parsed {len(variants_df)} variants from combined variant column")

variants_df['position'] = variants_df['position'].astype(int)

# Remove duplicates
n_before = len(variants_df)
variants_df = variants_df.drop_duplicates(
    subset=['gene', 'position', 'ref_aa', 'alt_aa']
).reset_index(drop=True)
n_removed = n_before - len(variants_df)
if n_removed > 0:
    print(f"  ⚠ Removed {n_removed} duplicate variant rows")

ann_cols = [c for c in variants_df.columns
            if c in ['alphamissense', 'franklin', 'alphamissense_pathogenicity']]
annotation_df = variants_df[['gene', 'position', 'ref_aa', 'alt_aa'] + ann_cols].copy()

print(f"✓ Loaded {len(variants_df)} variants across {variants_df['gene'].nunique()} genes")
print(variants_df.groupby('gene').size().to_string())


## Cell 4: Monomer Structural Metrics

In [ ]:
# =============================================================================
# CELL 4: MONOMER METRICS (v6.0)
# =============================================================================
# v6.0 changes:
#   1. Added ±3 neighborhood contact extraction with distance-weighted sum
#   2. pLDDT-gated flanking residues (≥50 to contribute, variant pos ≥50 for evaluability)
# =============================================================================

print("Extracting monomer structural metrics...")

# v6.0: Neighborhood weights (Spec 2.2.1)
NBHD_WEIGHTS = {0: 1.00, 1: 0.75, 2: 0.50, 3: 0.25}

def extract_neighborhood_monomer(contact_map, plddt_map, variant_pos):
    """
    Extract ±3 neighborhood contacts with distance-weighted sum (Spec 2.2.1-2.2.2).
    Flanking residues with pLDDT < 50 are zeroed out, not invalidating the window.
    Variant position must have pLDDT ≥ 50 for evaluability.
    """
    result = {
        'nbhd_mono_contacts_weighted': None,
        'nbhd_mono_contacts_raw': None,
        'nbhd_mono_evaluable': False,
        'nbhd_mono_n_eval_positions': 0,
    }
    if not contact_map or not plddt_map:
        return result

    variant_plddt = plddt_map.get(variant_pos)
    if variant_plddt is None or variant_plddt < 50:
        return result

    result['nbhd_mono_evaluable'] = True
    weighted_sum = 0.0
    raw_sum = 0.0
    n_eval = 0

    for offset in range(-3, 4):
        pos = variant_pos + offset
        weight = NBHD_WEIGHTS.get(abs(offset), 0)
        pos_plddt = plddt_map.get(pos)
        if pos_plddt is not None and pos_plddt >= 50:
            contacts = contact_map.get(pos, 0)
            weighted_sum += weight * contacts
            raw_sum += contacts
            n_eval += 1

    result['nbhd_mono_contacts_weighted'] = round(weighted_sum, 3)
    result['nbhd_mono_contacts_raw'] = raw_sum
    result['nbhd_mono_n_eval_positions'] = n_eval
    return result


monomer_rows = []

for gene in variants_df['gene'].unique():
    g = str(gene).lower()
    gene_vars = variants_df[variants_df['gene'].str.lower() == g]

    # Get pLDDT (CIF first for genes with zero PDB B-factors)
    plddt_map, struct_plddt, plddt_src, plddt_path = get_monomer_plddt(g)

    # Get PDB structure for contacts and accessibility
    _, pdb_name = MONOMER_STRUCTURES.get(g, (None, None))
    pdb_path = find_file(pdb_name)
    if pdb_path is None:
        for pat in [f'fold_{g}_model_0.pdb', f'{g}.pdb']:
            pdb_path = find_file(pat)
            if pdb_path: break

    struct_pdb = load_pdb(pdb_path)
    struct_contacts = struct_pdb or struct_plddt
    contact_map = count_contacts(struct_contacts, 'A') if struct_contacts else {}
    acc_map = get_accessibility(struct_pdb or struct_plddt, 'A')
    ss_map = get_secondary_structure(pdb_path, 'A')
    aa_map = get_residue_aa(struct_plddt or struct_pdb, 'A')

    n_plddt = sum(1 for _, r in gene_vars.iterrows() if plddt_map.get(int(r['position'])))
    n_acc = sum(1 for _, r in gene_vars.iterrows() if acc_map.get(int(r['position'])) is not None)
    has_struct = struct_plddt is not None or struct_pdb is not None
    print(f"  {gene}: struct={'YES' if has_struct else 'NO'} src={plddt_src} pLDDT={n_plddt}/{len(gene_vars)} acc={n_acc}/{len(gene_vars)} contacts={len(contact_map)} SS={len(ss_map)}")

    for _, row in gene_vars.iterrows():
        pos = int(row['position'])
        gd = get_grantham(row['ref_aa'], row['alt_aa'])
        p = plddt_map.get(pos)
        c = contact_map.get(pos, 0) if contact_map else None
        a = acc_map.get(pos)
        sec_struct = ss_map.get(pos, '-') if ss_map else None

        # v6.0: Neighborhood ±3 extraction
        nbhd = extract_neighborhood_monomer(contact_map, plddt_map, pos)

        monomer_rows.append({
            'gene': gene, 'position': pos,
            'ref_aa': row['ref_aa'], 'alt_aa': row['alt_aa'],
            'grantham_distance': gd, 'grantham_class': classify_grantham(gd),
            'substitution_severity': round(grantham_severity(gd), 2),
            'property_changes': get_property_changes(row['ref_aa'], row['alt_aa']),
            'monomer_plddt': p, 'monomer_plddt_category': classify_plddt(p),
            'monomer_n_contacts': float(c) if c is not None else None,
            'monomer_contact_category': classify_contacts(c),
            'monomer_aa': aa_map.get(pos),
            'monomer_accessibility': a,
            'monomer_burial': classify_burial(a),
            'monomer_secondary_structure': sec_struct,
            'monomer_contact_disruption': float(c) if c is not None else None,
            # v6.0: Neighborhood columns
            'nbhd_mono_contacts_weighted': nbhd['nbhd_mono_contacts_weighted'],
            'nbhd_mono_contacts_raw': nbhd['nbhd_mono_contacts_raw'],
            'nbhd_mono_evaluable': nbhd['nbhd_mono_evaluable'],
            'nbhd_mono_n_eval_positions': nbhd['nbhd_mono_n_eval_positions'],
        })

df = pd.DataFrame(monomer_rows)
print(f"\n✓ Monomer: {len(df)} variants")
print(f"  pLDDT populated:     {df['monomer_plddt'].notna().sum()}/{len(df)}")
print(f"  Accessibility:       {df['monomer_accessibility'].notna().sum()}/{len(df)}")
print(f"  Burial (non-unknown): {(df['monomer_burial'] != 'unknown').sum()}/{len(df)}")
print(f"  Secondary structure: {df['monomer_secondary_structure'].notna().sum()}/{len(df)}")
# v6.0 stats
print(f"  Neighborhood evaluable: {df['nbhd_mono_evaluable'].sum()}/{len(df)}")
print(f"  Nbhd weighted contacts (non-null): {df['nbhd_mono_contacts_weighted'].notna().sum()}/{len(df)}")


## Cell 5: Multimer Structural Metrics

In [ ]:
# =============================================================================
# CELL 5: MULTIMER EXTRACTION (v6.0)
# =============================================================================
# v6.0 changes:
#   1. CDH2 offset correction (truncated −159, cyto −745)
#   2. ±3 neighborhood contact extraction for multimer
#   3. Per-partner neighborhood inter-chain contacts and interface detection
# =============================================================================

print("Extracting multimer metrics...")

# CONSTRUCT_OFFSETS is defined in Cell 1

def apply_construct_offset(gene_lower, partner_label, full_length_pos):
    """Apply construct offset correction. Returns (corrected_pos, is_valid)."""
    offset = CONSTRUCT_OFFSETS.get(partner_label.lower())
    if offset is None:
        return full_length_pos, True
    corrected = full_length_pos - offset
    return (corrected, True) if corrected > 0 else (None, False)


def extract_neighborhood_multimer(contact_map, inter_map, plddt_map, variant_pos):
    """
    Extract ±3 neighborhood from multimer with distance-weighted sum (Spec 2.2.1).
    Includes both intra-chain and inter-chain contacts.
    """
    result = {
        'nbhd_multi_contacts_weighted': None,
        'nbhd_multi_inter_weighted': None,
        'nbhd_multi_evaluable': False,
        'nbhd_multi_has_interface': False,
    }
    if not contact_map or not plddt_map:
        return result
    variant_plddt = plddt_map.get(variant_pos)
    if variant_plddt is None or variant_plddt < 50:
        return result

    result['nbhd_multi_evaluable'] = True
    weighted_contacts = 0.0
    weighted_inter = 0.0

    for offset in range(-3, 4):
        pos = variant_pos + offset
        weight = NBHD_WEIGHTS.get(abs(offset), 0)
        pos_plddt = plddt_map.get(pos)
        if pos_plddt is not None and pos_plddt >= 50:
            weighted_contacts += weight * contact_map.get(pos, 0)
            inter = inter_map.get(pos, 0) if inter_map else 0
            weighted_inter += weight * inter
            if inter > 0:
                result['nbhd_multi_has_interface'] = True

    result['nbhd_multi_contacts_weighted'] = round(weighted_contacts, 3)
    result['nbhd_multi_inter_weighted'] = round(weighted_inter, 3)
    return result


multi_data = {}  # (gene_lower, position) → {col: val}
all_partner_labels = set()
variant_genes = set(df['gene'].str.lower().unique())

for gene1, partner_label, cif_file, pdb_file, chain1, chain2, is_primary in MULTIMER_STRUCTURES:
    g1 = gene1.lower()
    plabel = partner_label.lower()

    pdb_path = find_file(pdb_file)
    cif_path = find_file(cif_file) if cif_file else None

    if pdb_path is None:
        print(f"  ⚠ {pdb_file} NOT FOUND — skipping {g1}-{plabel}")
        continue

    struct_pdb = load_pdb(pdb_path)
    if struct_pdb is None:
        print(f"  ⚠ Failed to load {pdb_file}")
        continue

    # === FORWARD: gene1 variants get multi_{partner_label}_* ===
    if g1 in variant_genes:
        all_partner_labels.add(plabel)
        gene1_positions = set(df[df['gene'].str.lower() == g1]['position'].values)

        plddt_a, psrc = get_multimer_plddt(pdb_path, cif_path, chain1)
        contacts_a = count_contacts(struct_pdb, chain1)
        inter_a, iface_a = count_interface(struct_pdb, chain1, chain2)
        acc_a = get_accessibility(struct_pdb, chain1)
        ss_a = get_secondary_structure(pdb_path, chain1)

        n_offset_nulled = 0
        for pos in gene1_positions:
            key = (g1, pos)
            if key not in multi_data:
                multi_data[key] = {}
            pfx = f"multi_{plabel}"

            # v6.0: CDH2 offset correction
            corrected_pos, is_valid = apply_construct_offset(g1, plabel, pos)

            if not is_valid:
                # Position outside construct range — null everything
                for suffix in ['_plddt','_n_contacts','_inter_contacts','_is_interface',
                               '_accessibility','_burial','_sec_struct','_disruption',
                               '_nbhd_contacts_weighted','_nbhd_inter_weighted',
                               '_nbhd_evaluable','_nbhd_has_interface']:
                    multi_data[key][f"{pfx}{suffix}"] = None
                multi_data[key][f"{pfx}_is_interface"] = False
                multi_data[key][f"{pfx}_nbhd_evaluable"] = False
                multi_data[key][f"{pfx}_nbhd_has_interface"] = False
                n_offset_nulled += 1
                continue

            # Use corrected position for lookups
            p = plddt_a.get(corrected_pos)
            c = contacts_a.get(corrected_pos, 0)
            ic = inter_a.get(corrected_pos, 0)
            multi_data[key][f"{pfx}_plddt"] = p
            multi_data[key][f"{pfx}_n_contacts"] = float(c)
            multi_data[key][f"{pfx}_inter_contacts"] = float(ic)
            multi_data[key][f"{pfx}_is_interface"] = corrected_pos in iface_a
            multi_data[key][f"{pfx}_accessibility"] = acc_a.get(corrected_pos)
            multi_data[key][f"{pfx}_burial"] = classify_burial(acc_a.get(corrected_pos))
            multi_data[key][f"{pfx}_sec_struct"] = ss_a.get(corrected_pos, '-')
            multi_data[key][f"{pfx}_disruption"] = float(c)

            # v6.0: Neighborhood ±3
            nbhd = extract_neighborhood_multimer(contacts_a, inter_a, plddt_a, corrected_pos)
            multi_data[key][f"{pfx}_nbhd_contacts_weighted"] = nbhd['nbhd_multi_contacts_weighted']
            multi_data[key][f"{pfx}_nbhd_inter_weighted"] = nbhd['nbhd_multi_inter_weighted']
            multi_data[key][f"{pfx}_nbhd_evaluable"] = nbhd['nbhd_multi_evaluable']
            multi_data[key][f"{pfx}_nbhd_has_interface"] = nbhd['nbhd_multi_has_interface']

        n_fwd = len([p for p in gene1_positions if plddt_a.get(apply_construct_offset(g1, plabel, p)[0] or -1)])
        offset_msg = f" ({n_offset_nulled} nulled by CDH2 offset)" if n_offset_nulled > 0 else ""
        print(f"  FWD {g1} → multi_{plabel}: {n_fwd}/{len(gene1_positions)} pLDDT (src={psrc}){offset_msg}")

    # === REVERSE: partner gene variants get multi_{partner_label}_* ===
    partner_gene_map = {
        'dvl2':'dvl2','ctnnb1':'ctnnb1','rock2':'rock2',
        'gli3':'gli3','kpna1':'kpna1','kpna6':'kpna6',
        'mdfi':'mdfi','tcf7l1':'tcf7l1',
        'cdh2_truncated':'cdh2','cdh2_cyto':'cdh2',
        'actin':'actb','actb':'actb','actb_no_bind':'actb',
    }
    partner_gene = partner_gene_map.get(plabel, plabel)

    if partner_gene in variant_genes:
        rev_label = plabel
        all_partner_labels.add(rev_label)
        partner_positions = set(df[df['gene'].str.lower() == partner_gene]['position'].values)

        plddt_b, psrc_b = get_multimer_plddt(pdb_path, cif_path, chain2)
        contacts_b = count_contacts(struct_pdb, chain2)
        inter_b, iface_b = count_interface(struct_pdb, chain2, chain1)
        acc_b = get_accessibility(struct_pdb, chain2)
        ss_b = get_secondary_structure(pdb_path, chain2)

        n_offset_nulled_rev = 0
        for pos in partner_positions:
            key = (partner_gene, pos)
            if key not in multi_data:
                multi_data[key] = {}
            pfx = f"multi_{rev_label}"
            # Only write if not already populated
            if f"{pfx}_plddt" not in multi_data[key] or multi_data[key][f"{pfx}_plddt"] is None:
                # v6.0: CDH2 offset correction for reverse
                corrected_pos, is_valid = apply_construct_offset(partner_gene, rev_label, pos)

                if not is_valid:
                    for suffix in ['_plddt','_n_contacts','_inter_contacts','_is_interface',
                                   '_accessibility','_burial','_sec_struct','_disruption',
                                   '_nbhd_contacts_weighted','_nbhd_inter_weighted',
                                   '_nbhd_evaluable','_nbhd_has_interface']:
                        multi_data[key][f"{pfx}{suffix}"] = None
                    multi_data[key][f"{pfx}_is_interface"] = False
                    multi_data[key][f"{pfx}_nbhd_evaluable"] = False
                    multi_data[key][f"{pfx}_nbhd_has_interface"] = False
                    n_offset_nulled_rev += 1
                    continue

                p = plddt_b.get(corrected_pos)
                c = contacts_b.get(corrected_pos, 0)
                ic = inter_b.get(corrected_pos, 0)
                multi_data[key][f"{pfx}_plddt"] = p
                multi_data[key][f"{pfx}_n_contacts"] = float(c)
                multi_data[key][f"{pfx}_inter_contacts"] = float(ic)
                multi_data[key][f"{pfx}_is_interface"] = corrected_pos in iface_b
                multi_data[key][f"{pfx}_accessibility"] = acc_b.get(corrected_pos)
                multi_data[key][f"{pfx}_burial"] = classify_burial(acc_b.get(corrected_pos))
                multi_data[key][f"{pfx}_sec_struct"] = ss_b.get(corrected_pos, '-')
                multi_data[key][f"{pfx}_disruption"] = float(c)

                # v6.0: Neighborhood ±3
                nbhd = extract_neighborhood_multimer(contacts_b, inter_b, plddt_b, corrected_pos)
                multi_data[key][f"{pfx}_nbhd_contacts_weighted"] = nbhd['nbhd_multi_contacts_weighted']
                multi_data[key][f"{pfx}_nbhd_inter_weighted"] = nbhd['nbhd_multi_inter_weighted']
                multi_data[key][f"{pfx}_nbhd_evaluable"] = nbhd['nbhd_multi_evaluable']
                multi_data[key][f"{pfx}_nbhd_has_interface"] = nbhd['nbhd_multi_has_interface']

        n_rev = len([p for p in partner_positions if plddt_b.get(apply_construct_offset(partner_gene, rev_label, p)[0] or -1)])
        offset_msg_rev = f" ({n_offset_nulled_rev} nulled by CDH2 offset)" if n_offset_nulled_rev > 0 else ""
        print(f"  REV {partner_gene} → multi_{rev_label}: {n_rev}/{len(partner_positions)} pLDDT (src={psrc_b}){offset_msg_rev}")

# Merge into df
multi_df = pd.DataFrame.from_dict(multi_data, orient='index')
multi_df.index = pd.MultiIndex.from_tuples(multi_df.index, names=['gene_lower','position'])
multi_df = multi_df.reset_index()

df['gene_lower'] = df['gene'].str.lower()
df = df.merge(multi_df, on=['gene_lower','position'], how='left')
df = df.drop(columns=['gene_lower'])

# === Summary columns ===
def compute_summary(row):
    partners, plddt_v, contact_v, disrupt_v, iface_partners = [], [], [], [], []
    for pl in all_partner_labels:
        p_col = f"multi_{pl}_plddt"
        if p_col in row.index and pd.notna(row[p_col]):
            partners.append(pl)
            plddt_v.append(row[p_col])
            c = sf(row.get(f"multi_{pl}_n_contacts"), 0)
            contact_v.append(c)
            d = sf(row.get(f"multi_{pl}_disruption"), 0)
            disrupt_v.append(d)
            if sb(row.get(f"multi_{pl}_is_interface"), False):
                iface_partners.append(pl)
    return pd.Series({
        'n_multimer_complexes': len(partners),
        'multimer_partners': ';'.join(partners) if partners else None,
        'is_interface_any': len(iface_partners) > 0,
        'interface_partners': ';'.join(iface_partners) if iface_partners else None,
        'n_interface_partners': len(iface_partners),
        'multimer_plddt_max': max(plddt_v) if plddt_v else None,
        'multimer_plddt_avg': round(np.mean(plddt_v), 2) if plddt_v else None,
        'multimer_contacts_max': max(contact_v) if contact_v else None,
        'multimer_contacts_avg': round(np.mean(contact_v), 2) if contact_v else None,
        'multimer_disruption_max': max(disrupt_v) if disrupt_v else None,
        'multimer_disruption_avg': round(np.mean(disrupt_v), 2) if disrupt_v else None,
    })

summary = df.apply(compute_summary, axis=1)
df = pd.concat([df, summary], axis=1)

# best_plddt
def best_p(row):
    vals = []
    if pd.notna(row.get('monomer_plddt')): vals.append(row['monomer_plddt'])
    for pl in all_partner_labels:
        col = f"multi_{pl}_plddt"
        if col in row.index and pd.notna(row[col]): vals.append(row[col])
    return max(vals) if vals else None

df['best_plddt'] = df.apply(best_p, axis=1)
df['confidence'] = df['best_plddt'].apply(lambda x: 'high' if pd.notna(x) and x >= 70 else ('low' if pd.notna(x) else 'unknown'))

# v6.0: CDH2 offset verification
# Offset verification
if CONSTRUCT_OFFSETS:
    print("\n=== Construct Offset Verification ===")
    for pl, off in CONSTRUCT_OFFSETS.items():
        print(f"  {pl}: offset={off}")
    print("=== End verification ===")


print(f"\n✓ Multimer complete (v6.0)")
print(f"  Interface variants: {df['is_interface_any'].sum()}/{len(df)}")
print(f"  With multimer data: {df['n_multimer_complexes'].gt(0).sum()}/{len(df)}")
print(f"  best_plddt filled:  {df['best_plddt'].notna().sum()}/{len(df)}")


## Cell 6: FoldX ΔΔG Loading (Three-Axis Framework)

In [ ]:
# =============================================================================
# CELL 6: FOLDX DDG (v6.0-A3: Three-Axis DDG Framework)
# =============================================================================
# Three independent DDG axes:
#   Axis 1: ddg_monomer — intrinsic fold stability (gated by monomer_plddt)
#   Axis 2: ddg_fold_{partner} — fold stability in complex context
#   Axis 3: ddg_binding_{partner} — interaction energy change (PPI)
#
# Changes from A2:
#   1. Monomer DDG gated by monomer_plddt (not best_plddt)
#   2. Per-partner: both ddg_fold and ddg_binding loaded
#   3. ddg_interpretation per-partner and variant-level summary
#   4. Aggregates recomputed from ddg_binding (consistent methodology)
#   5. Partner name normalization (carried from A2)
#   6. ddg_multimer_min NaN reconstruction (carried from A2)
# =============================================================================

# DDG mechanism boundaries
DDG_DESTAB = 1.0
DDG_HIGHLY = 2.0

PER_PARTNER_DDG_LABELS = ALL_PARTNER_LABELS

# ─────────────────────────────────────────────────────────────────────────────
# Partner name normalization (A2.4)
# ─────────────────────────────────────────────────────────────────────────────

def normalize_foldx_partner(partner_name, gene, position=None):
    """Normalize FoldX partner name to structural extraction label."""
    p = partner_name.strip().lower()
    g = gene.strip().lower()
    if p == 'cdh2':
        return 'cdh2_truncated'
    if p == 'shroom3':
        return {'rock2':'rock2','dvl2':'dvl2','cdh2':'cdh2_truncated','ctnnb1':'ctnnb1'}.get(g, p)
    if p == 'zic3' and g == 'gli3':
        return 'gli3'
    return p

def normalize_partners_tested(partners_str, gene, position=None):
    if pd.isna(partners_str) or not str(partners_str).strip():
        return partners_str
    parts = [p.strip() for p in str(partners_str).split(';') if p.strip()]
    normalized = [normalize_foldx_partner(p, gene, position) for p in parts]
    seen = set()
    result = []
    for p in normalized:
        if p not in seen:
            seen.add(p)
            result.append(p)
    return ';'.join(result)

# ─────────────────────────────────────────────────────────────────────────────
# DDG classification function
# ─────────────────────────────────────────────────────────────────────────────

def classify_ddg(v):
    """Classify a DDG value into a category."""
    if pd.isna(v): return None
    v = float(v)
    if v > 2.0: return 'highly_destabilizing'
    elif v > 1.0: return 'destabilizing'
    elif v > 0.5: return 'mildly_destabilizing'
    elif v > -0.5: return 'neutral'
    elif v > -1.0: return 'mildly_stabilizing'
    elif v > -2.0: return 'stabilizing'
    else: return 'highly_stabilizing'

# ─────────────────────────────────────────────────────────────────────────────
# Load monomer DDG
# ─────────────────────────────────────────────────────────────────────────────
if MONOMER_DDG_FILE.exists():
    ddg = pd.read_csv(MONOMER_DDG_FILE)
    ddg.columns = [c.lower() for c in ddg.columns]
    dc = next((c for c in ['ddg','ddg_monomer','total_ddg'] if c in ddg.columns), None)
    if dc:
        ddg = ddg.rename(columns={dc:'ddg_monomer'})
        if 'ddg_monomer' in df.columns:
            df = df.drop(columns=['ddg_monomer'])
        df = df.merge(ddg[['gene','position','ref_aa','alt_aa','ddg_monomer']],
                      on=['gene','position','ref_aa','alt_aa'], how='left')
        print(f"✓ Monomer DDG: {df['ddg_monomer'].notna().sum()}/{len(df)}")
    else:
        print(f"⚠ Monomer DDG file: no recognized column in {list(ddg.columns)}")
else:
    if 'ddg_monomer' not in df.columns:
        df['ddg_monomer'] = None
    print(f"⚠ Monomer DDG file not found (existing: {df['ddg_monomer'].notna().sum()}/{len(df)})")

# ─────────────────────────────────────────────────────────────────────────────
# Load multimer DDG (aggregate — for backwards compat and min fix)
# ─────────────────────────────────────────────────────────────────────────────
if MULTIMER_DDG_FILE.exists():
    ddgm = pd.read_csv(MULTIMER_DDG_FILE)
    ddgm.columns = [c.lower() for c in ddgm.columns]
    dc = next((c for c in ['ddg','ddg_multimer','total_ddg'] if c in ddgm.columns), None)
    if dc:
        grp = ddgm.groupby(['gene','position','ref_aa','alt_aa'])
        agg = grp.agg(
            ddg_multimer_max=(dc,'max'), ddg_multimer_min=(dc,'min'),
            ddg_multimer_mean=(dc,'mean'), n_complexes_tested=(dc,'count')
        ).reset_index()
        if 'partner' in ddgm.columns:
            pt = grp['partner'].apply(lambda x: ';'.join(x.astype(str))).reset_index()
            pt.columns = ['gene','position','ref_aa','alt_aa','partners_tested']
            agg = agg.merge(pt, on=['gene','position','ref_aa','alt_aa'], how='left')
        for c in agg.columns:
            if c in df.columns and c not in ['gene','position','ref_aa','alt_aa']:
                df = df.drop(columns=[c])
        df = df.merge(agg, on=['gene','position','ref_aa','alt_aa'], how='left')
        print(f"✓ Multimer DDG (raw): {df['ddg_multimer_max'].notna().sum()}/{len(df)}")
    elif 'ddg_multimer_max' in ddgm.columns:
        merge_cols = ['gene','position','ref_aa','alt_aa']
        avail = [c for c in ['ddg_multimer_max','ddg_multimer_min','ddg_multimer_mean',
                             'n_complexes_tested','partners_tested'] if c in ddgm.columns]
        for c in avail:
            if c in df.columns:
                df = df.drop(columns=[c])
        df = df.merge(ddgm[merge_cols + avail], on=merge_cols, how='left')
        print(f"✓ Multimer DDG (pre-aggregated): {df['ddg_multimer_max'].notna().sum()}/{len(df)}")
else:
    for c in ['ddg_multimer_max','ddg_multimer_min','ddg_multimer_mean',
              'n_complexes_tested','partners_tested']:
        if c not in df.columns:
            df[c] = None
    existing = df['ddg_multimer_max'].notna().sum()
    if existing > 0:
        print(f"  Using existing multimer DDG: {existing}/{len(df)}")

for c in ['ddg_monomer','ddg_multimer_max','ddg_multimer_min','ddg_multimer_mean',
          'n_complexes_tested','partners_tested']:
    if c not in df.columns: df[c] = None

# ─────────────────────────────────────────────────────────────────────────────
# Fix ddg_multimer_min NaN (A2.3)
# ─────────────────────────────────────────────────────────────────────────────
_n_min_fixed = 0
for idx in df.index:
    if pd.notna(df.at[idx, 'ddg_multimer_max']) and pd.isna(df.at[idx, 'ddg_multimer_min']):
        n = df.at[idx, 'n_complexes_tested']
        mx = float(df.at[idx, 'ddg_multimer_max'])
        mn = df.at[idx, 'ddg_multimer_mean']
        if pd.notna(n) and pd.notna(mn):
            n, mn = int(n), float(mn)
            if n == 1: df.at[idx, 'ddg_multimer_min'] = mx
            elif n == 2: df.at[idx, 'ddg_multimer_min'] = round(2 * mn - mx, 6)
            else: df.at[idx, 'ddg_multimer_min'] = mn
            _n_min_fixed += 1
print(f"✓ A2.3: Fixed {_n_min_fixed} ddg_multimer_min NaN values")

# ─────────────────────────────────────────────────────────────────────────────
# Normalize partners_tested (A2.4)
# ─────────────────────────────────────────────────────────────────────────────
_n_norm = 0
for idx in df.index:
    old_pt = df.at[idx, 'partners_tested']
    if pd.notna(old_pt):
        new_pt = normalize_partners_tested(old_pt, df.at[idx, 'gene'], df.at[idx, 'position'])
        if new_pt != old_pt:
            df.at[idx, 'partners_tested'] = new_pt
            _n_norm += 1
print(f"✓ A2.4: Normalized {_n_norm} partners_tested entries")

# ─────────────────────────────────────────────────────────────────────────────
# A3: Monomer DDG confidence — gated by monomer_plddt (not best_plddt)
# ─────────────────────────────────────────────────────────────────────────────

def assign_monomer_ddg_confidence(monomer_plddt):
    if pd.isna(monomer_plddt): return 'unknown'
    p = float(monomer_plddt)
    if p >= 70: return 'high'
    elif p >= 50: return 'moderate'
    else: return 'low'

df['ddg_confidence'] = df['monomer_plddt'].apply(assign_monomer_ddg_confidence)
df['ddg_monomer_confident'] = df['monomer_plddt'].apply(
    lambda x: True if pd.notna(x) and float(x) >= 50 else False)

def classify_ddg_with_confidence(row):
    raw_cat = classify_ddg(row.get('ddg_monomer'))
    if raw_cat is None: return None
    if not row.get('ddg_monomer_confident', False):
        return raw_cat + '_unreliable'
    return raw_cat

df['ddg_category_raw'] = df['ddg_monomer'].apply(classify_ddg)
df['ddg_category'] = df.apply(classify_ddg_with_confidence, axis=1)

_mono_conf = df['ddg_monomer_confident'].sum()
print(f"\n✓ Monomer DDG confidence (gated by monomer_plddt):")
print(f"  Confident: {_mono_conf}, Unreliable: {df['ddg_monomer'].notna().sum() - _mono_conf}")
print(df['ddg_confidence'].value_counts().to_string())

# ─────────────────────────────────────────────────────────────────────────────
# A3 Phase B: Load per-partner DDG (both fold and binding)
# ─────────────────────────────────────────────────────────────────────────────

# Initialize all per-partner columns
for pl in PER_PARTNER_DDG_LABELS:
    df[f"ddg_binding_{pl}"] = np.nan
    df[f"ddg_fold_{pl}"] = np.nan
    df[f"ddg_binding_category_{pl}"] = None
    df[f"ddg_fold_category_{pl}"] = None
    df[f"ddg_{pl}_confident"] = False
    df[f"ddg_interp_{pl}"] = None

PER_PARTNER_DDG_FILE = WORKING_DIR / "results" / "foldx_ddg_per_partner.csv"
if not PER_PARTNER_DDG_FILE.exists():
    PER_PARTNER_DDG_FILE = WORKING_DIR / "foldx_ddg_per_partner.csv"

_pp_loaded = 0
_pp_confident = 0

if PER_PARTNER_DDG_FILE.exists():
    pp_df = pd.read_csv(PER_PARTNER_DDG_FILE)
    pp_df['gene'] = pp_df['gene'].astype(str).str.lower()
    pp_df['partner'] = pp_df['partner'].astype(str).str.lower()
    print(f"\n  Loading per-partner DDG from {PER_PARTNER_DDG_FILE}")
    print(f"  Rows: {len(pp_df)}, with data: {pp_df['ddg_multimer'].notna().sum()}")

    for _, pp_row in pp_df.iterrows():
        if pd.isna(pp_row.get('ddg_binding')) and pd.isna(pp_row.get('ddg_fold')):
            continue

        gene = pp_row['gene']
        pos = pp_row['position']
        ref = pp_row['ref_aa']
        alt = pp_row['alt_aa']
        partner = pp_row['partner']

        mask = (df['gene'] == gene) & (df['position'] == pos) & \
               (df['ref_aa'] == ref) & (df['alt_aa'] == alt)
        matches = df.index[mask]
        if len(matches) == 0:
            continue
        idx = matches[0]

        # Check columns exist
        if f"ddg_binding_{partner}" not in df.columns:
            continue

        # Per-partner pLDDT confidence
        plddt_col = f"multi_{partner}_plddt"
        plddt_val = df.at[idx, plddt_col] if plddt_col in df.columns else np.nan
        is_confident = pd.notna(plddt_val) and float(plddt_val) >= 50
        df.at[idx, f"ddg_{partner}_confident"] = is_confident

        # Load ddg_binding
        bind_val = pp_row.get('ddg_binding')
        if pd.notna(bind_val):
            bind_val = float(bind_val)
            df.at[idx, f"ddg_binding_{partner}"] = bind_val
            raw = classify_ddg(bind_val)
            df.at[idx, f"ddg_binding_category_{partner}"] = \
                (raw + '_unreliable' if raw and not is_confident else raw)

        # Load ddg_fold
        fold_val = pp_row.get('ddg_fold')
        if pd.notna(fold_val):
            fold_val = float(fold_val)
            df.at[idx, f"ddg_fold_{partner}"] = fold_val
            raw = classify_ddg(fold_val)
            df.at[idx, f"ddg_fold_category_{partner}"] = \
                (raw + '_unreliable' if raw and not is_confident else raw)

        # Per-partner interpretation
        mono_ddg = df.at[idx, 'ddg_monomer']
        if pd.notna(mono_ddg) and pd.notna(bind_val):
            mono_v = float(mono_ddg)
            mono_dir = "disrupted" if mono_v > DDG_DESTAB else ("stabilized" if mono_v < -DDG_DESTAB else "neutral")
            bind_dir = "disrupted" if bind_val > DDG_DESTAB else ("strengthened" if bind_val < -DDG_DESTAB else "neutral")
            fold_context = ""
            if pd.notna(fold_val):
                fold_multi_dir = "disrupted" if fold_val > DDG_DESTAB else ("stabilized" if fold_val < -DDG_DESTAB else "neutral")
                if mono_dir != fold_multi_dir:
                    if mono_dir == "disrupted" and fold_multi_dir in ("neutral","stabilized"):
                        fold_context = "|fold_context_rescued"
                    elif mono_dir in ("neutral","stabilized") and fold_multi_dir == "disrupted":
                        fold_context = "|fold_context_worsened"
                    elif mono_dir == "disrupted" and fold_multi_dir == "disrupted":
                        fold_context = "|fold_context_confirmed"
                    else:
                        fold_context = f"|fold_context_{fold_multi_dir}"
                elif mono_dir == "disrupted" and fold_multi_dir == "disrupted":
                    fold_context = "|fold_context_confirmed"
            interp = f"{mono_dir}_fold/{bind_dir}_interaction{fold_context}"
            df.at[idx, f"ddg_interp_{partner}"] = interp

        _pp_loaded += 1
        if is_confident:
            _pp_confident += 1

    print(f"\n✓ Phase B: Per-partner DDG loaded (fold + binding)")
    print(f"  Loaded: {_pp_loaded} variant-partner pairs")
    print(f"  Confident (pLDDT ≥ 50): {_pp_confident}")
    for pl in PER_PARTNER_DDG_LABELS:
        n_b = df[f"ddg_binding_{pl}"].notna().sum()
        n_f = df[f"ddg_fold_{pl}"].notna().sum()
        if n_b > 0 or n_f > 0:
            n_c = df[f"ddg_{pl}_confident"].sum()
            print(f"    {pl}: bind={n_b}, fold={n_f}, confident={n_c}")
else:
    print(f"\n⚠ Per-partner DDG file not found at {PER_PARTNER_DDG_FILE}")
    print(f"  Run foldx_parse_per_partner.py to generate this file")

# ─────────────────────────────────────────────────────────────────────────────
# Recompute aggregates from ddg_binding (consistent methodology)
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n  Recomputing aggregates from ddg_binding...")
_n_recomputed = 0
for idx in df.index:
    bind_vals = []
    bind_partners = []
    for pl in PER_PARTNER_DDG_LABELS:
        v = df.at[idx, f"ddg_binding_{pl}"]
        if pd.notna(v):
            bind_vals.append(float(v))
            bind_partners.append(pl)

    if bind_vals:
        df.at[idx, 'ddg_multimer_max'] = max(bind_vals)
        df.at[idx, 'ddg_multimer_min'] = min(bind_vals)
        df.at[idx, 'ddg_multimer_mean'] = round(sum(bind_vals) / len(bind_vals), 6)
        df.at[idx, 'n_complexes_tested'] = len(bind_vals)
        df.at[idx, 'partners_tested'] = ';'.join(bind_partners)
        _n_recomputed += 1

print(f"  Recomputed aggregates for {_n_recomputed} variants (from ddg_binding)")

# ─────────────────────────────────────────────────────────────────────────────
# ddg_category_multimer (from ddg_binding, per-partner gated)
# ─────────────────────────────────────────────────────────────────────────────

# NOTE: DDG summary, low-confidence flags, and ddg_category_multimer
# are computed in Cell 9 after all DDG data is loaded (Cells 6, 6a, 6b).

print(f"\n✓ DDG framework complete (v6.0-A3)")
print(f"  Monomer DDG: {df['ddg_monomer'].notna().sum()} ({df['ddg_monomer_confident'].sum()} confident)")
print(f"  Per-partner binding values: {sum(df[f'ddg_binding_{pl}'].notna().sum() for pl in PER_PARTNER_DDG_LABELS)}")
print(f"  Per-partner fold values: {sum(df[f'ddg_fold_{pl}'].notna().sum() for pl in PER_PARTNER_DDG_LABELS)}")
print(f"  → Summary, categories, and flags computed in Cell 9 after all DDG data loaded")


## Cell 6a: FoldX Monomer ΔΔG Runner

**Requires FoldX 5.**

In [ ]:
# =============================================================================
# CELL 6a: FOLDX MONOMER DDG FOR MISSING VARIANTS (v6.0-A3-6)
# =============================================================================
# A3-6 changes:
#   Fix 1: RepairPDB before BuildModel (cached per structure)
#   Fix 2: Improved Dif file detection (exact match first, then targeted glob)
# =============================================================================

import shutil

# FOLDX_BINARY is configured in Cell 1
FOLDX_DIR_BASE = WORKING_DIR / "foldx_runs"
FOLDX_DIR_BASE.mkdir(parents=True, exist_ok=True)

# rotabase.txt should be in the FoldX binary directory
ROTABASE = FOLDX_BINARY.parent / "rotabase.txt" if FOLDX_BINARY.exists() else Path("rotabase.txt")

ONE_TO_THREE = {v:k for k,v in THREE_TO_ONE.items()}

# ─────────────────────────────────────────────────────────────────────────────
# A3-6 Fix 1: RepairPDB cache
# ─────────────────────────────────────────────────────────────────────────────
_REPAIRED_CACHE = {}  # pdb_path → repaired_pdb_path

def repair_pdb(structure_path, work_dir):
    """Run FoldX RepairPDB and return path to repaired structure.
    Results are cached per input file to avoid redundant computation.
    Returns original path if RepairPDB fails (graceful fallback)."""
    structure_path = Path(structure_path)
    
    # Check cache
    cache_key = str(structure_path.resolve())
    if cache_key in _REPAIRED_CACHE:
        cached = _REPAIRED_CACHE[cache_key]
        if cached.exists():
            return cached
    
    work_dir = Path(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)
    struct_name = structure_path.name
    
    # Copy structure + rotabase to work dir
    shutil.copy2(structure_path, work_dir / struct_name)
    if ROTABASE.exists() and not (work_dir / "rotabase.txt").exists():
        shutil.copy2(ROTABASE, work_dir / "rotabase.txt")
    
    cmd = [
        str(FOLDX_BINARY),
        "--command=RepairPDB",
        f"--pdb={struct_name}",
        f"--output-dir={work_dir}",
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=600, cwd=str(work_dir))
        if result.returncode != 0:
            print(f"    RepairPDB warning: non-zero exit for {struct_name}, using original")
            _REPAIRED_CACHE[cache_key] = structure_path
            return structure_path
        
        # FoldX names repaired files as: {basename}_Repair.pdb
        base = struct_name.replace('.pdb', '')
        repaired = work_dir / f"{base}_Repair.pdb"
        if not repaired.exists():
            # Fallback: search for any *_Repair.pdb
            for f in work_dir.glob("*_Repair.pdb"):
                repaired = f
                break
        
        if repaired.exists():
            _REPAIRED_CACHE[cache_key] = repaired
            return repaired
        else:
            print(f"    RepairPDB warning: no repaired file found for {struct_name}, using original")
            _REPAIRED_CACHE[cache_key] = structure_path
            return structure_path
    
    except subprocess.TimeoutExpired:
        print(f"    RepairPDB timeout for {struct_name}, using original")
        _REPAIRED_CACHE[cache_key] = structure_path
        return structure_path
    except Exception as e:
        print(f"    RepairPDB exception for {struct_name}: {e}, using original")
        _REPAIRED_CACHE[cache_key] = structure_path
        return structure_path


def run_foldx_buildmodel(structure_path, chain_id, ref_aa, position, alt_aa, work_dir, n_runs=3):
    """Run FoldX BuildModel and return DDG. Returns None on failure.
    
    A3-6: Uses RepairPDB-processed structure for improved baseline accuracy.
    """
    work_dir = Path(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)

    # A3-6 Fix 1: Repair structure before BuildModel
    repair_dir = FOLDX_DIR_BASE / "repaired"
    repair_dir.mkdir(parents=True, exist_ok=True)
    repaired_path = repair_pdb(structure_path, repair_dir)
    
    # Copy repaired structure to work dir
    struct_name = repaired_path.name
    shutil.copy2(repaired_path, work_dir / struct_name)

    # Copy rotabase if needed
    if ROTABASE.exists() and not (work_dir / "rotabase.txt").exists():
        shutil.copy2(ROTABASE, work_dir / "rotabase.txt")

    # FoldX mutation format: {wt_aa_1letter}{chain}{position}{mut_aa_1letter};
    mut_str = f"{ref_aa}{chain_id}{position}{alt_aa};"

    # Write individual_list.txt
    mut_file = work_dir / "individual_list.txt"
    mut_file.write_text(mut_str + "\n")

    # Run FoldX
    cmd = [
        str(FOLDX_BINARY),
        "--command=BuildModel",
        f"--pdb={struct_name}",
        "--mutant-file=individual_list.txt",
        f"--numberOfRuns={n_runs}",
        f"--output-dir={work_dir}",
    ]

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300, cwd=str(work_dir))
        if result.returncode != 0:
            print(f"    FoldX error: {result.stderr[:200]}")
            return None

        # A3-6 Fix 2: Improved Dif file detection
        # Try exact match first, then targeted glob
        struct_base = struct_name.replace('.pdb', '')
        dif_file = work_dir / f"Dif_{struct_base}.fxout"
        if not dif_file.exists():
            # Targeted fallback: only match Dif files for this structure
            candidates = list(work_dir.glob(f"Dif_{struct_base}*.fxout"))
            if not candidates:
                candidates = list(work_dir.glob("Dif_*.fxout"))
            if candidates:
                dif_file = candidates[0]

        if not dif_file.exists():
            print(f"    No Dif output found in {work_dir}")
            return None

        # Read DDG: skip header lines, take average of runs
        ddg_values = []
        with open(dif_file) as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('Pdb') or line.startswith('#'):
                    continue
                parts = line.split('\t')
                if len(parts) >= 2:
                    try:
                        ddg_values.append(float(parts[1]))
                    except ValueError:
                        continue

        if ddg_values:
            return round(sum(ddg_values) / len(ddg_values), 4)
        return None

    except subprocess.TimeoutExpired:
        print(f"    FoldX timeout for {ref_aa}{position}{alt_aa}")
        return None
    except Exception as e:
        print(f"    FoldX exception: {e}")
        return None

# Identify variants missing monomer DDG
missing_mono = df[df['ddg_monomer'].isna()].copy()
print(f"Variants missing monomer DDG: {len(missing_mono)}")
if len(missing_mono) > 0:
    print(f"  Genes: {missing_mono['gene'].value_counts().to_string()}")

# Verify FoldX binary exists
if not FOLDX_BINARY.exists():
    print(f"⚠ FoldX binary not found at {FOLDX_BINARY}")
    print("  Skipping FoldX computation. Set FOLDX_BINARY path and re-run this cell.")
else:
    new_ddg_mono = {}
    for idx, row in missing_mono.iterrows():
        gene = str(row['gene']).lower()
        pos = int(row['position'])
        ref = str(row['ref_aa'])
        alt = str(row['alt_aa'])

        # Find PDB structure
        _, pdb_name = MONOMER_STRUCTURES.get(gene, (None, None))
        pdb_path = find_file(pdb_name)
        if pdb_path is None:
            print(f"  ⚠ No monomer PDB for {gene} — skipping {ref}{pos}{alt}")
            continue

        work_dir = FOLDX_DIR_BASE / "monomer" / f"{gene}_{ref}{pos}{alt}"
        print(f"  Running FoldX: {gene} {ref}{pos}{alt}...", end=" ")
        ddg = run_foldx_buildmodel(pdb_path, 'A', ref, pos, alt, work_dir)
        if ddg is not None:
            new_ddg_mono[idx] = ddg
            print(f"DDG = {ddg:.2f}")
        else:
            print("FAILED")

    # Merge new DDG values
    for idx, ddg in new_ddg_mono.items():
        df.loc[idx, 'ddg_monomer'] = ddg
        df.loc[idx, 'ddg_category'] = classify_ddg(ddg)

    print(f"\n✓ Computed {len(new_ddg_mono)} new monomer DDG values")
    print(f"  Total monomer DDG coverage: {df['ddg_monomer'].notna().sum()}/{len(df)}")


## Cell 6b: FoldX Multimer ΔΔG Runner

**Requires FoldX 5 and Cell 6a.**

In [ ]:
# =============================================================================
# CELL 6b: FOLDX MULTIMER DDG FOR MISSING VARIANTS (v6.0-A3-6)
# =============================================================================
# A3-6 changes:
#   Fix 1: RepairPDB before BuildModel (uses shared cache from cell 6a)
#   Fix 2: Tightened mutant PDB glob (exact match + prefix-filtered fallback)
#   Fix 3: Idempotent re-run protection (reset aggregates, skip existing pairs)
#   Fix 4: Bidirectional complex lookup table (correct chain assignment for
#           variants in genes appearing as chain B)
# =============================================================================

# For multimer DDG, run BuildModel on each complex PDB where the variant
# is at the interface, then compute interaction energy difference via AnalyseComplex.

def run_foldx_analysecomplex(structure_path, chains, work_dir):
    """Run FoldX AnalyseComplex and return interaction energy."""
    work_dir = Path(work_dir)
    struct_name = structure_path.name
    if not (work_dir / struct_name).exists():
        shutil.copy2(structure_path, work_dir / struct_name)
    if ROTABASE.exists() and not (work_dir / "rotabase.txt").exists():
        shutil.copy2(ROTABASE, work_dir / "rotabase.txt")

    cmd = [
        str(FOLDX_BINARY),
        "--command=AnalyseComplex",
        f"--pdb={struct_name}",
        f"--analyseComplexChains={chains}",
        f"--output-dir={work_dir}",
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300, cwd=str(work_dir))
        if result.returncode != 0:
            return None

        # Parse Interaction_{name}_AC.fxout
        for f in work_dir.glob("Interaction_*_AC.fxout"):
            with open(f) as fh:
                for line in fh:
                    if line.startswith('Pdb') or line.startswith('#') or not line.strip():
                        continue
                    parts = line.strip().split('\t')
                    if len(parts) >= 6:
                        try:
                            return float(parts[5])  # Interaction Energy
                        except ValueError:
                            continue
        return None
    except Exception:
        return None


def run_foldx_multimer_ddg(pdb_path, chain_gene, chain_partner, ref_aa, position, alt_aa, work_dir, n_runs=3):
    """Compute multimer DDG: RepairPDB → BuildModel → AnalyseComplex on WT and mutant.
    
    A3-6: Uses RepairPDB-processed structure; tightened mutant PDB detection.
    """
    work_dir = Path(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)
    chains_str = f"{chain_gene},{chain_partner}"

    # A3-6 Fix 1: Repair structure before BuildModel
    repair_dir = FOLDX_DIR_BASE / "repaired"
    repair_dir.mkdir(parents=True, exist_ok=True)
    repaired_path = repair_pdb(pdb_path, repair_dir)

    # Step 1: AnalyseComplex on wild-type (repaired)
    wt_dir = work_dir / "wt"
    wt_dir.mkdir(exist_ok=True)
    ie_wt = run_foldx_analysecomplex(repaired_path, chains_str, wt_dir)

    # Step 2: BuildModel to get mutant structure (uses repaired input)
    ddg_fold = run_foldx_buildmodel(repaired_path, chain_gene, ref_aa, position, alt_aa, work_dir, n_runs)

    # A3-6 Fix 2: Tightened mutant PDB detection
    # FoldX5 names mutant structures as {basename}_1.pdb (run 1)
    struct_base = repaired_path.stem
    mutant_pdb = work_dir / f"{struct_base}_1.pdb"
    
    if not mutant_pdb.exists():
        # Prefix-filtered fallback: only match files starting with our structure name
        candidates = [f for f in work_dir.glob(f"{struct_base}_*.pdb")
                      if f.name != repaired_path.name  # exclude WT copy
                      and '_Repair' not in f.name]      # exclude repair artifacts
        if candidates:
            # Pick the first run file (numerically lowest suffix)
            candidates.sort()
            mutant_pdb = candidates[0]
        else:
            mutant_pdb = None

    ie_mut = None
    if mutant_pdb is not None and mutant_pdb.exists():
        mut_dir = work_dir / "mut"
        mut_dir.mkdir(exist_ok=True)
        ie_mut = run_foldx_analysecomplex(mutant_pdb, chains_str, mut_dir)

    # DDG_binding = IE(mutant) - IE(wt)
    ddg_binding = None
    if ie_wt is not None and ie_mut is not None:
        ddg_binding = round(ie_mut - ie_wt, 4)

    return ddg_fold, ddg_binding


# ─────────────────────────────────────────────────────────────────────────────
# A3-6 Fix 4: Bidirectional complex lookup table
# ─────────────────────────────────────────────────────────────────────────────
# Build a proper mapping: (variant_gene, partner_label) → (pdb_file, my_chain, partner_chain)
# This handles both forward (gene is chain A) and reverse (gene is chain B) lookups.

_COMPLEX_LOOKUP = {}  # (gene_lower, partner_label) → (pdb_file, my_chain, partner_chain)

# Map partner labels to the actual gene name they represent
_PARTNER_TO_GENE = {
    'dvl2':'dvl2', 'ctnnb1':'ctnnb1', 'rock2':'rock2',
    'gli3':'gli3', 'kpna1':'kpna1', 'kpna6':'kpna6',
    'mdfi':'mdfi', 'tcf7l1':'tcf7l1',
    'cdh2_truncated':'cdh2', 'cdh2_cyto':'cdh2',
    'actin':'actb', 'actb':'actb', 'actb_no_bind':'actb',
}

for _g1, _pl, _cif, _pdb, _c1, _c2, _primary in MULTIMER_STRUCTURES:
    _g1 = _g1.lower()
    _pl_lower = _pl.lower()
    
    # Forward: gene1 variant in this complex, partner is _pl
    _COMPLEX_LOOKUP[(_g1, _pl_lower)] = (_pdb, _c1, _c2)
    
    # Reverse: partner gene variant in this complex, interacting with gene1
    _partner_gene = _PARTNER_TO_GENE.get(_pl_lower, _pl_lower)
    if _partner_gene != _g1:  # avoid self-mapping
        # For the reverse, the variant gene is _partner_gene, it sits on chain _c2,
        # and its interaction partner (gene1) sits on chain _c1.
        # The "partner label" from the variant's perspective is gene1.
        # But we key by the original partner_label since that's what interface_partners uses.
        _reverse_key = (_partner_gene, _pl_lower)
        if _reverse_key not in _COMPLEX_LOOKUP:
            _COMPLEX_LOOKUP[_reverse_key] = (_pdb, _c2, _c1)

print(f"✓ A3-6: Built bidirectional complex lookup ({len(_COMPLEX_LOOKUP)} entries)")
for k, v in sorted(_COMPLEX_LOOKUP.items()):
    print(f"    {k[0]} × {k[1]} → {v[0]} (chains {v[1]},{v[2]})")


# ─────────────────────────────────────────────────────────────────────────────
# Run multimer DDG for missing variants
# ─────────────────────────────────────────────────────────────────────────────

if not FOLDX_BINARY.exists():
    print("⚠ FoldX binary not found — skipping multimer DDG")
else:
    # Variants at interfaces that don't have multimer DDG
    missing_multi = df[df['is_interface_any'] & df['ddg_multimer_max'].isna()].copy()
    print(f"Interface variants missing multimer DDG: {len(missing_multi)}")

    # A3-6 Fix 3: Track already-computed pairs to avoid re-run accumulation
    _computed_pairs = set()
    
    new_multi_results = []

    for idx, row in missing_multi.iterrows():
        gene = str(row['gene']).lower()
        pos = int(row['position'])
        ref = str(row['ref_aa'])
        alt = str(row['alt_aa'])
        partners = str(row.get('interface_partners', ''))
        if not partners or partners == 'nan':
            continue

        for partner in partners.split(';'):
            partner = partner.strip()
            if not partner:
                continue
            
            # A3-6 Fix 3: Skip already-computed pairs
            pair_key = (gene, pos, ref, alt, partner)
            if pair_key in _computed_pairs:
                continue

            # A3-6 Fix 4: Use bidirectional lookup
            lookup_key = (gene, partner.lower())
            matched = _COMPLEX_LOOKUP.get(lookup_key)
            
            if matched is None:
                # Try with normalized partner name
                norm_partner = normalize_foldx_partner(partner, gene, pos)
                lookup_key = (gene, norm_partner.lower())
                matched = _COMPLEX_LOOKUP.get(lookup_key)
            
            if matched is None:
                print(f"  ⚠ No complex found for {gene} × {partner}")
                continue

            pdb_file, my_chain, partner_chain = matched
            pdb_path = find_file(pdb_file)
            if pdb_path is None:
                print(f"  ⚠ PDB not found: {pdb_file}")
                continue

            # v6.0: CDH2 offset correction for DDG queries (Spec 12.4)
            query_pos = pos
            if gene == 'cdh2':
                cdh2_offset = CONSTRUCT_OFFSETS.get(partner.lower(), 0)
                if cdh2_offset > 0:
                    query_pos = pos - cdh2_offset
                    if query_pos <= 0:
                        print(f"  ⚠ CDH2 {ref}{pos}{alt} vs {partner}: "
                              f"position outside construct range (offset={cdh2_offset})")
                        continue

            work_dir = FOLDX_DIR_BASE / "multimer" / f"{gene}_{ref}{pos}{alt}_{partner}"
            print(f"  Running FoldX multimer: {gene} {ref}{query_pos}{alt} × {partner} "
                  f"(orig pos {pos}, chains {my_chain},{partner_chain})...", end=" ")

            ddg_fold, ddg_binding = run_foldx_multimer_ddg(
                pdb_path, my_chain, partner_chain, ref, query_pos, alt, work_dir
            )

            _computed_pairs.add(pair_key)

            ddg_val = ddg_binding if ddg_binding is not None else ddg_fold
            if ddg_val is not None:
                new_multi_results.append({
                    'idx': idx, 'gene': gene, 'position': pos,
                    'ref_aa': ref, 'alt_aa': alt, 'partner': partner,
                    'ddg_multimer': ddg_val, 'ddg_fold': ddg_fold, 'ddg_binding': ddg_binding
                })
                print(f"DDG_binding={ddg_binding}, DDG_fold={ddg_fold}")
            else:
                print("FAILED")

    # A3-6 Fix 3: Reset aggregate columns before recomputation
    # This ensures re-running the cell produces clean results
    if new_multi_results:
        multi_new = pd.DataFrame(new_multi_results)
        grp = multi_new.groupby('idx').agg(
            ddg_max=('ddg_multimer', 'max'),
            ddg_mean=('ddg_multimer', 'mean'),
            n_tested=('ddg_multimer', 'count'),
            partners=('partner', lambda x: ';'.join(x))
        )
        for idx, row in grp.iterrows():
            # Overwrite (not accumulate) for idempotent re-run
            df.loc[idx, 'ddg_multimer_max'] = row['ddg_max']
            df.loc[idx, 'ddg_multimer_mean'] = row['ddg_mean']
            df.loc[idx, 'n_complexes_tested'] = row['n_tested']
            df.loc[idx, 'partners_tested'] = row['partners']
            # min follows from the data
            if row['n_tested'] == 1:
                df.loc[idx, 'ddg_multimer_min'] = row['ddg_max']
            else:
                vals = multi_new[multi_new['idx'] == idx]['ddg_multimer'].values
                df.loc[idx, 'ddg_multimer_min'] = min(vals)

        print(f"\n✓ Computed {len(new_multi_results)} new multimer DDG values across {len(grp)} variants")
    else:
        print("  No new multimer DDG computed")

    # Save expanded DDG results for future re-use
    foldx_out = RESULTS_DIR / "foldx_ddg_expanded_results.csv"
    ddg_cols_save = ['gene','position','ref_aa','alt_aa','ddg_monomer','ddg_category',
                     'ddg_multimer_max','ddg_multimer_mean','n_complexes_tested','partners_tested']
    df[[c for c in ddg_cols_save if c in df.columns]].to_csv(foldx_out, index=False)
    print(f"  Saved expanded DDG to {foldx_out}")

print(f"\nFinal DDG coverage:")
print(f"  Monomer DDG: {df['ddg_monomer'].notna().sum()}/{len(df)}")
print(f"  Multimer DDG: {df['ddg_multimer_max'].notna().sum()}/{len(df)}")


## Cell 7: Scoring, Tier Assignment, and Mechanism Classification

In [ ]:
# =============================================================================
# CELL 9: SCORING, TIERS, MECHANISM (v6.0-A2)
# =============================================================================
# v6.0-A2 changes (Addendum A2):
#   1. 16-category mechanism framework (3 new conflicting-signal categories)
#   2. PPI DDG evaluation uses corrected min + normalized partners
#   3. Low-confidence DDG flagging (ddg_low_confidence_flag)
#   4. Unchanged: Disruption scoring, tier thresholds, evaluability
# =============================================================================

# ALL_PARTNER_LABELS is set dynamically in Cell 5

# Tier thresholds
TIER_THRESHOLDS = [(5.0, "Tier 1"), (3.0, "Tier 2"), (1.5, "Tier 3")]
DEFAULT_TIER = "Tier 4"

# DDG mechanism boundaries
DDG_DESTAB = 1.0
DDG_HIGHLY = 2.0

# Disruption → points
DISRUPTION_POINTS = [(20, 4.0), (10, 3.0), (4, 2.0), (1, 1.0)]

# Contact threshold for mechanism split (75th percentile)
CONTACT_DRIVEN_THRESHOLD = 6

BURIAL_RANK = {'unknown': 0, 'surface_exposed': 1, 'partially_buried': 2, 'buried_core': 3}
RANK_TO_BURIAL = {v: k for k, v in BURIAL_RANK.items()}



# ─────────────────────────────────────────────────────────────────────────────
# 
# ─────────────────────────────────────────────────────────────────────────────
# Deduplication check
# ─────────────────────────────────────────────────────────────────────────────
_n_before = len(df)
_n_dupes = df.duplicated(['gene','position','ref_aa','alt_aa']).sum()
if _n_dupes > 0:
    print(f'⚠ {_n_dupes} duplicate rows detected — deduplicating (keeping first)...')
    _duped = df[df.duplicated(['gene','position','ref_aa','alt_aa'], keep=False)]
    for (g,p,r,a), grp in _duped.groupby(['gene','position','ref_aa','alt_aa']):
        print(f'  {g} {r}{int(p)}{a}: {len(grp)} rows')
    df = df.drop_duplicates(['gene','position','ref_aa','alt_aa'], keep='first').reset_index(drop=True)
    print(f'  Reduced: {_n_before} → {len(df)} rows')
del _n_before, _n_dupes

# A2.3 INLINE: ddg_multimer_min NaN repair (runs before mechanism assignment)
# ─────────────────────────────────────────────────────────────────────────────
# This MUST run here — after ALL data loading/merging (Cells 6, 6a, 6b) and
# before mechanism classification which depends on ddg_multimer_min.
_n_min_before = df['ddg_multimer_min'].isna().sum()
_n_multi = df['ddg_multimer_max'].notna().sum()
_n_fixed = 0

for _idx in df.index:
    _max_v = df.at[_idx, 'ddg_multimer_max']
    _min_v = df.at[_idx, 'ddg_multimer_min']
    if pd.notna(_max_v) and pd.isna(_min_v):
        _n_t = df.at[_idx, 'n_complexes_tested']
        _mx = float(_max_v)
        _mn = df.at[_idx, 'ddg_multimer_mean']
        if pd.notna(_n_t) and pd.notna(_mn):
            _n_t = int(_n_t)
            _mn = float(_mn)
            if _n_t == 1:
                df.at[_idx, 'ddg_multimer_min'] = _mx
            elif _n_t == 2:
                df.at[_idx, 'ddg_multimer_min'] = round(2 * _mn - _mx, 6)
            else:
                df.at[_idx, 'ddg_multimer_min'] = _mn
            _n_fixed += 1

_remaining = ((df['ddg_multimer_max'].notna()) & (df['ddg_multimer_min'].isna())).sum()
print(f"A2.3: ddg_multimer_min repair: {_n_fixed} fixed out of {_n_min_before} NaN "
      f"({_n_multi} multimer variants). Remaining NaN: {_remaining}")
if _remaining > 0:
    print(f"  ⚠ WARNING: {_remaining} NaN mins could not be fixed!")
del _n_min_before, _n_multi, _n_fixed, _remaining

# ddg_category_multimer computed below



# ─────────────────────────────────────────────────────────────────────────────
# A3: Recompute ddg_category_multimer (needs corrected min from above)
# ─────────────────────────────────────────────────────────────────────────────

def classify_ddg_multimer_final(row):
    confident_vals = []
    all_vals = []
    for pl in ALL_PARTNER_LABELS:
        v = row.get(f"ddg_binding_{pl}")
        if pd.notna(v):
            all_vals.append(float(v))
            if row.get(f"ddg_{pl}_confident", False):
                confident_vals.append(float(v))
    if confident_vals:
        extreme = max(confident_vals, key=abs)
        return classify_ddg(extreme), classify_ddg(extreme)
    if all_vals:
        extreme = max(all_vals, key=abs)
        raw = classify_ddg(extreme)
        return (raw + '_unreliable' if raw else None), raw
    mx = row.get('ddg_multimer_max')
    mn = row.get('ddg_multimer_min')
    agg = []
    if pd.notna(mx): agg.append(float(mx))
    if pd.notna(mn): agg.append(float(mn))
    if not agg: return None, None
    extreme = max(agg, key=abs)
    raw = classify_ddg(extreme)
    plddt_max = row.get('multimer_plddt_max')
    if pd.notna(plddt_max) and float(plddt_max) < 50:
        return (raw + '_unreliable' if raw else None), raw
    return raw, raw

_mc = df.apply(classify_ddg_multimer_final, axis=1)
df['ddg_category_multimer'] = [r[0] for r in _mc]
df['ddg_category_multimer_raw'] = [r[1] for r in _mc]
print(f"  ddg_category_multimer: {df['ddg_category_multimer'].notna().sum()}")

# ─────────────────────────────────────────────────────────────────────────────
# A3: Variant-level DDG summary (runs here so all DDG data is loaded)
# ─────────────────────────────────────────────────────────────────────────────

def compute_ddg_summary(row):
    mono = row.get('ddg_monomer')
    mono_conf = row.get('ddg_monomer_confident', False)

    if pd.notna(mono) and mono_conf:
        mv = float(mono)
        mono_status = "fold_disrupted" if mv > DDG_DESTAB else ("fold_stabilized" if mv < -DDG_DESTAB else "fold_neutral")
    elif pd.notna(mono):
        mono_status = "fold_unknown(low_confidence)"
    else:
        mono_status = "fold_unknown"

    flags = [mono_status]
    affected = []
    neutral = []

    for pl in ALL_PARTNER_LABELS:
        bind_v = row.get(f"ddg_binding_{pl}")
        is_conf = row.get(f"ddg_{pl}_confident", False)
        if pd.isna(bind_v) or not is_conf:
            continue
        bv = float(bind_v)
        if abs(bv) > DDG_DESTAB:
            affected.append(f"{pl}({bv:+.1f})")
        else:
            neutral.append(pl)

    fold_rescued = []
    fold_worsened = []
    for pl in ALL_PARTNER_LABELS:
        fold_v = row.get(f"ddg_fold_{pl}")
        is_conf = row.get(f"ddg_{pl}_confident", False)
        if pd.isna(fold_v) or not is_conf:
            continue
        fv = float(fold_v)
        if pd.notna(mono) and mono_conf:
            mv = float(mono)
            mono_dir = "disrupted" if mv > DDG_DESTAB else ("stabilized" if mv < -DDG_DESTAB else "neutral")
            fold_dir = "disrupted" if fv > DDG_DESTAB else ("stabilized" if fv < -DDG_DESTAB else "neutral")
            if mono_dir == "disrupted" and fold_dir in ("neutral", "stabilized"):
                fold_rescued.append(pl)
            elif mono_dir in ("neutral", "stabilized") and fold_dir == "disrupted":
                fold_worsened.append(pl)

    has_strengthened = any("(-" in a for a in affected)
    has_disrupted = any("(+" in a for a in affected)

    if has_disrupted:
        flags.append("has_interaction_disrupted")
    if has_strengthened:
        flags.append("has_interaction_strengthened")
    if has_disrupted and has_strengthened:
        flags.append("mixed_interaction_effects")
    if fold_rescued:
        flags.append(f"fold_context_rescued({';'.join(fold_rescued)})")
    if fold_worsened:
        flags.append(f"fold_context_worsened({';'.join(fold_worsened)})")

    return (
        '; '.join(flags),
        ', '.join(affected) if affected else '',
        ', '.join(neutral) if neutral else '',
    )

_summ = df.apply(compute_ddg_summary, axis=1)
df['ddg_summary_flags'] = [r[0] for r in _summ]
df['ddg_summary_partners_affected'] = [r[1] for r in _summ]
df['ddg_summary_partners_neutral'] = [r[2] for r in _summ]
_n_affected = (df['ddg_summary_partners_affected'] != '').sum()
print(f"  DDG summary: {_n_affected} variants with affected partners")

# ─────────────────────────────────────────────────────────────────────────────
# A3: Low-confidence DDG flagging
# ─────────────────────────────────────────────────────────────────────────────

def compute_low_confidence_flag(row):
    flags = []
    for pl in ALL_PARTNER_LABELS:
        bind_v = row.get(f"ddg_binding_{pl}")
        if pd.isna(bind_v):
            continue
        is_conf = row.get(f"ddg_{pl}_confident", False)
        if not is_conf and abs(float(bind_v)) > DDG_HIGHLY:
            plddt_col = f"multi_{pl}_plddt"
            plddt = row.get(plddt_col)
            plddt_str = f"{int(plddt)}" if pd.notna(plddt) else "?"
            flags.append(f"{pl}:{float(bind_v):.2f}(pLDDT={plddt_str})")
    if not flags:
        has_pp = any(pd.notna(row.get(f"ddg_binding_{pl}")) for pl in ALL_PARTNER_LABELS)
        if not has_pp:
            multi_plddt_max = sf(row.get('multimer_plddt_max'), 0)
            if 0 < multi_plddt_max < 50:
                for col_name, label in [('ddg_multimer_min', 'multi_min'),
                                         ('ddg_multimer_max', 'multi_max'),
                                         ('ddg_multimer_mean', 'multi_mean')]:
                    v = row.get(col_name)
                    if pd.notna(v) and abs(float(v)) > DDG_HIGHLY:
                        flags.append(f"aggregate_{label}:{float(v):.2f}(pLDDT={int(multi_plddt_max)})")
    return ';'.join(flags) if flags else ''

df['ddg_low_confidence_flag'] = df.apply(compute_low_confidence_flag, axis=1)
_n_flagged = (df['ddg_low_confidence_flag'] != '').sum()
print(f"  Low-confidence DDG flags: {_n_flagged}")


def compute_score_v6(row):
    """Pipeline 1 structural disruption score (v6.0). Unchanged from v6.0."""
    score, ev = 0.0, []

    # DISRUPTION: severity × (mono + MAX pLDDT-gated inter)
    sev = sf(row.get('substitution_severity'), 0)
    mono_c = sf(row.get('monomer_n_contacts'), 0)

    max_inter = 0.0
    best_inter_partner = None
    gated_iface_partners = []

    for pl in ALL_PARTNER_LABELS:
        plddt_col = f"multi_{pl}_plddt"
        ic_col = f"multi_{pl}_inter_contacts"
        iface_col = f"multi_{pl}_is_interface"

        pl_plddt = sf(row.get(plddt_col), 0) if plddt_col in row.index else 0
        if pl_plddt < 50:
            continue

        if iface_col in row.index and sb(row.get(iface_col)):
            gated_iface_partners.append(pl)

        if ic_col in row.index and pd.notna(row[ic_col]):
            ic_val = float(row[ic_col])
            if ic_val > max_inter:
                max_inter = ic_val
                best_inter_partner = pl

    total_contacts = mono_c + max_inter
    disruption = round(sev * total_contacts, 2)

    for thresh, pts in DISRUPTION_POINTS:
        if disruption >= thresh:
            score += pts
            ev.append(f'disruption({disruption:.1f})')
            break
    else:
        ev.append(f'no_disruption({disruption:.1f})')

    # INTERFACE BONUS (pLDDT-gated)
    n_gated = len(gated_iface_partners)
    if n_gated >= 2:
        score += 2.0;
        ev.append(f'multi_interface({n_gated})')
    elif n_gated == 1:
        score += 1.5;
        ev.append(f'interface({gated_iface_partners[0]})')

    # BURIAL (best across mono + confident multimer)
    mono_burial = ss(row.get('monomer_burial'))
    best_rank = BURIAL_RANK.get(mono_burial, 0)
    best_source = 'monomer'
    for pl in ALL_PARTNER_LABELS:
        burial_col = f"multi_{pl}_burial"
        plddt_col = f"multi_{pl}_plddt"
        if burial_col in row.index and pd.notna(row.get(burial_col)):
            if sf(row.get(plddt_col), 0) >= 50:
                pl_rank = BURIAL_RANK.get(ss(row[burial_col]), 0)
                if pl_rank > best_rank:
                    best_rank = pl_rank
                    best_source = pl
    best_burial = RANK_TO_BURIAL.get(best_rank, 'unknown')
    if best_burial == 'buried_core':
        score += 2.0;
        ev.append(f'buried_core({best_source})')
    elif best_burial == 'partially_buried':
        score += 1.0;
        ev.append(f'partially_buried({best_source})')

    # pLDDT MULTIPLIER
    bp = row.get('best_plddt')
    if bp is not None and not pd.isna(bp):
        bp_val = float(bp)
        if bp_val < 50:
            score *= 0.4;
            ev.append(f'very_low_plddt({int(bp_val)})')
        elif bp_val < 70:
            score *= 0.7;
            ev.append(f'low_plddt({int(bp_val)})')

    return pd.Series({
        'v6_final_score': round(score, 2),
        'v6_score_evidence': ';'.join(ev),
        'v6_contact_disruption': disruption,
        'v6_total_contacts': total_contacts,
        'v6_max_inter_contacts': max_inter,
        'v6_best_inter_partner': best_inter_partner or '',
        'v6_best_burial': best_burial,
        'v6_best_burial_source': best_source,
        'v6_interface_partners_gated': ';'.join(gated_iface_partners) if gated_iface_partners else '',
        'v6_n_interface_partners_gated': n_gated,
    })


def assign_tier(score):
    if pd.isna(score):
        return DEFAULT_TIER
    s = float(score)
    for thresh, label in TIER_THRESHOLDS:
        if s >= thresh:
            return label
    return DEFAULT_TIER


# === Evaluability flags ===
def compute_evaluability(df):
    df['structure_evaluable'] = df['best_plddt'].apply(
        lambda x: True if pd.notna(x) and float(x) >= 50 else False)
    df['ddg_evaluable'] = df['ddg_confidence'].apply(
        lambda x: ss(x).lower() != 'low' and ss(x) != '')
    df['am_evaluable'] = df['AlphaMissense'].notna() if 'AlphaMissense' in df.columns else False
    df['franklin_evaluable'] = df['franklin'].notna() if 'franklin' in df.columns else False

    def check_multi(row):
        for pl in ALL_PARTNER_LABELS:
            col = f"multi_{pl}_plddt"
            if col in row.index and pd.notna(row[col]) and float(row[col]) >= 50:
                return True
        return False

    df['multimer_evaluable'] = df.apply(check_multi, axis=1)
    return df


# === Franklin standardization (needed by mechanism & concordance) ===
def std_franklin(v):
    if pd.isna(v): return 'No data'
    v = str(v).strip();
    vl = v.lower()
    if 'pathogenic' in vl and 'likely' in vl:
        return 'Likely Pathogenic'
    elif 'pathogenic' in vl:
        return 'Pathogenic'
    elif 'benign' in vl and 'likely' in vl:
        return 'Likely Benign'
    elif 'benign' in vl:
        return 'Benign'
    elif 'vus' in vl and 'high' in vl:
        return 'VUS (high)'
    elif 'vus' in vl and 'mid' in vl:
        return 'VUS (mid)'
    elif 'vus' in vl and 'low' in vl:
        return 'VUS (low)'
    elif 'vus' in vl:
        return 'VUS'
    return v


# ─────────────────────────────────────────────────────────────────────────────
# A2.6: 16-CATEGORY MECHANISM CLASSIFICATION
# ─────────────────────────────────────────────────────────────────────────────
# 
# Destabilizing group (mono DDG > 1.0):
#   1  Fold + PPI destabilization
#   2  Fold destab. at interface
#   3  Fold destabilization
#   4  PPI destabilization (mono neutral)
#   5  Fold destab. + PPI stabilization (conflicting)    ← NEW
#
# Stabilizing group (mono DDG < -1.0):
#   6  Stabilizing + PPI (potential GoF)
#   7  Stabilizing at interface (potential GoF)
#   8  Stabilizing (potential GoF)
#   9  Stabilizing + PPI destabilization (conflicting)   ← NEW
#
# Neutral group (|mono DDG| ≤ 1.0):
#  10  PPI stabilization (potential GoF)
#  11  Interface variant (DDG neutral)
#  12  Structural variant - contact-driven (DDG neutral)
#  13  PPI conflicting (mixed partner signals)           ← NEW
#  14  Structural variant - burial-driven (DDG neutral)
#  15  Benign (structurally evaluated)
#  16  Structure unevaluable
# ─────────────────────────────────────────────────────────────────────────────

def classify_mechanism_v6(row):
    """16-category mechanism classification (v6.0-A3).
    
    Fold axis: uses BOTH ddg_monomer (gated by monomer_plddt) AND
    ddg_fold_{partner} (gated by partner pLDDT). Fold disruption fires
    if EITHER measure shows it.
    
    PPI axis: uses ddg_binding_{partner} (gated by partner pLDDT).
    Fires on confident |ddg_binding| > 1.0 regardless of interface status.
    """
    tier = ss(row.get('v6_tier'))
    is_high_tier = tier in ('Tier 1', 'Tier 2')
    structure_eval = row.get('structure_evaluable', False)

    ddg_m = row.get('ddg_monomer')
    mono_conf = row.get('ddg_monomer_confident', False)

    mono_c = sf(row.get('monomer_n_contacts'), 0)
    gated_iface = ss(row.get('v6_interface_partners_gated'))
    is_iface = len(gated_iface) > 0

    # ── Fold axis: monomer OR any confident partner fold ──
    has_dm = pd.notna(ddg_m)
    dm_val = float(ddg_m) if has_dm else 0.0

    # Monomer fold (gated by monomer_plddt)
    mono_fold_destab = has_dm and mono_conf and dm_val > DDG_DESTAB
    mono_fold_stab = has_dm and mono_conf and dm_val < -DDG_DESTAB

    # Partner-context fold (gated by per-partner pLDDT)
    partner_fold_destab = False
    partner_fold_stab = False
    for _pl in ALL_PARTNER_LABELS:
        _fv = row.get(f"ddg_fold_{_pl}")
        _conf = row.get(f"ddg_{_pl}_confident", False)
        if pd.notna(_fv) and _conf:
            if float(_fv) > DDG_DESTAB:
                partner_fold_destab = True
            if float(_fv) < -DDG_DESTAB:
                partner_fold_stab = True

    # Union: fold disrupted if either source shows it
    fold_destab = mono_fold_destab or partner_fold_destab
    fold_stab = mono_fold_stab or partner_fold_stab
    fold_neutral = not fold_destab and not fold_stab

    # ── PPI axis: ddg_binding per-partner (confident only) ──
    ppi_destab = False
    ppi_stab = False
    ppi_partner_destab = None
    ppi_partner_stab = None

    # Phase B: per-partner DDG_binding
    _has_per_partner = False
    for _pl in ALL_PARTNER_LABELS:
        _bv = row.get(f"ddg_binding_{_pl}")
        if pd.notna(_bv):
            _has_per_partner = True
            _conf = row.get(f"ddg_{_pl}_confident", False)
            if not _conf:
                continue
            _bv = float(_bv)
            if _bv > DDG_DESTAB:
                ppi_destab = True
                if ppi_partner_destab is None:
                    ppi_partner_destab = _pl
            if _bv < -DDG_DESTAB:
                ppi_stab = True
                if ppi_partner_stab is None:
                    ppi_partner_stab = _pl

    # Fallback to aggregate for variants without per-partner data
    if not _has_per_partner:
        ddg_x = row.get('ddg_multimer_max')
        ddg_n = row.get('ddg_multimer_min')
        pt = ss(row.get('partners_tested'))
        tested_partners = set(p.strip() for p in pt.split(';') if p.strip()) if pt else set()
        iface_partners = set(gated_iface.split(';')) if gated_iface else set()

        if tested_partners:
            has_dx = pd.notna(ddg_x)
            has_dn = pd.notna(ddg_n)
            dx_val = float(ddg_x) if has_dx else 0.0
            dn_val = float(ddg_n) if has_dn else 0.0
            overlap = iface_partners & tested_partners if is_iface else set()

            multi_destab = (has_dx and dx_val > DDG_DESTAB) or (has_dn and dn_val > DDG_DESTAB)
            multi_stab = (has_dx and dx_val < -DDG_DESTAB) or (has_dn and dn_val < -DDG_DESTAB)

            if multi_destab and (overlap or is_iface):
                ppi_destab = True
                ppi_partner_destab = sorted(overlap)[0] if overlap else (sorted(iface_partners)[0] if iface_partners else '')
            if multi_stab and (overlap or is_iface):
                ppi_stab = True
                ppi_partner_stab = sorted(overlap)[0] if overlap else (sorted(iface_partners)[0] if iface_partners else '')

    ppi_partner = ppi_partner_destab or ppi_partner_stab or ''

    # ── #16: Structure unevaluable ──
    if not structure_eval:
        am = ss(row.get('AlphaMissense')).lower()
        fr = std_franklin(row.get('franklin')).lower()
        ext_flag = (am == 'likely_pathogenic') or (fr in ('pathogenic', 'likely pathogenic', 'vus (high)'))
        return 'Structure unevaluable', ppi_partner, ext_flag

    # ── Fold disrupted group ──
    if fold_destab:
        if ppi_destab and ppi_stab:
            return 'Fold + PPI destabilization', ppi_partner_destab or '', False
        elif ppi_destab:
            return 'Fold + PPI destabilization', ppi_partner_destab or '', False
        elif ppi_stab:
            return 'Fold destab. + PPI stabilization (conflicting)', ppi_partner_stab or '', False
        elif is_iface:
            return 'Fold destab. at interface', '', False
        else:
            return 'Fold destabilization', '', False

    # ── Fold stabilized group ──
    if fold_stab:
        if ppi_stab and ppi_destab:
            return 'Stabilizing + PPI (potential GoF)', ppi_partner_stab or '', False
        elif ppi_stab:
            return 'Stabilizing + PPI (potential GoF)', ppi_partner_stab or '', False
        elif ppi_destab:
            return 'Stabilizing + PPI destabilization (conflicting)', ppi_partner_destab or '', False
        elif is_iface:
            return 'Stabilizing at interface (potential GoF)', '', False
        else:
            return 'Stabilizing (potential GoF)', '', False

    # ── Fold neutral group ──
    if ppi_destab and not ppi_stab:
        return 'PPI destabilization', ppi_partner_destab or '', False
    if ppi_stab and not ppi_destab:
        return 'PPI stabilization (potential GoF)', ppi_partner_stab or '', False
    if ppi_destab and ppi_stab:
        return 'PPI conflicting (mixed partner signals)', ppi_partner, False
    if is_iface:
        return 'Interface variant (DDG neutral)', '', False
    if is_high_tier:
        if mono_c >= CONTACT_DRIVEN_THRESHOLD:
            return 'Structural variant - contact-driven (DDG neutral)', '', False
        else:
            return 'Structural variant - burial-driven (DDG neutral)', '', False

    return 'Benign (structurally evaluated)', '', False


# === Apply scoring ===
print("Computing v6.0-A2 scores...")
score_results = df.apply(compute_score_v6, axis=1)
for c in score_results.columns:
    df[c] = score_results[c]
df['v6_tier'] = df['v6_final_score'].apply(assign_tier)

# Show tier changes from v5.2
if 'tier' in df.columns:
    df['_t52'] = df['tier'].str.extract(r'(Tier \d)')
    changed = df[df['_t52'] != df['v6_tier']]
    print(f"\nTier changes from v5.2: {changed.shape[0]}")
    for _, r in changed.iterrows():
        print(f"  {r['gene']} {r['ref_aa']}{int(r['position'])}{r['alt_aa']}: "
              f"{r['_t52']} → {r['v6_tier']} "
              f"(score {sf(r.get('final_score')):.2f} → {r['v6_final_score']:.2f})")
    df.drop(columns=['_t52'], inplace=True)

print(f"\n✓ v6.0-A2 Tier distribution:")
print(df['v6_tier'].value_counts().to_string())

# Evaluability
df = compute_evaluability(df)
print(f"\n✓ Evaluability: structure={df['structure_evaluable'].sum()}, "
      f"ddg={df['ddg_evaluable'].sum()}, multimer={df['multimer_evaluable'].sum()}")

# Mechanism (16-category)
mech_results = df.apply(classify_mechanism_v6, axis=1)
df['v6_mechanism'] = [r[0] for r in mech_results]
df['v6_mechanism_partner'] = [r[1] for r in mech_results]
df['v6_external_evidence_flag'] = [r[2] for r in mech_results]

print(f"\n✓ v6.0-A2 Mechanism distribution (16-category):")
print(df['v6_mechanism'].value_counts().to_string())





## Cell 8: Concordance, Pipeline 2, and Pipeline Agreement

In [ ]:
# =============================================================================
# CELL 10: CONCORDANCE + PIPELINE 2 + AGREEMENT (v6.0-A2)
# =============================================================================
# v6.0 changes:
#   1. Three concordance flavors: strict, relaxed, T3-inclusive
#   2. Adjusted denominators (X/N where N = evaluable axes)
#   3. Pipeline 2: Neighborhood ±3 scoring with real extraction data
#   4. Pipeline agreement tracking
#   5. Legacy three-way concordance dropped
#   6. Class A-J dropped; variant_summary string replaces it
# v6.0-A2 changes:
#   7. max_abs_ddg now includes corrected ddg_multimer_min (A2.3 fix)
#   8. Neighborhood concordance + mechanism uses same 16-category framework
# =============================================================================

# === Merge annotations from annotation_df (loaded in Cell 3) ===
# Drop existing annotation columns first to prevent duplicates on re-merge
_ann_drop = [c for c in ['AlphaMissense', 'AlphaMissense_pathogenicity',
                         'AlphaMissense_raw', 'franklin', 'franklin_raw',
                         'alphamissense', 'alphamissense_pathogenicity']
             if c in df.columns]
if _ann_drop:
    df = df.drop(columns=_ann_drop)
    print(f"  Dropped existing annotation columns before re-merge: {_ann_drop}")

df['gene_lower'] = df['gene'].astype(str).str.lower()
if annotation_df is not None:
    ann = annotation_df.copy()
    ann.columns = [c.lower() for c in ann.columns]
    ann['gene_lower'] = ann['gene'].astype(str).str.lower()
    for col in ['alphamissense', 'alphamissense_pathogenicity', 'franklin']:
        if col in ann.columns:
            m = ann[['gene_lower', 'position', 'ref_aa', 'alt_aa', col]].drop_duplicates()
            df = df.merge(m, on=['gene_lower', 'position', 'ref_aa', 'alt_aa'], how='left')
            rn = {'alphamissense': 'AlphaMissense', 'alphamissense_pathogenicity': 'AlphaMissense_pathogenicity'}
            if col in rn:
                df = df.rename(columns={col: rn[col]})
            print(f"  Merged {col}: {df[rn.get(col, col)].notna().sum()}/{len(df)}")
df = df.drop(columns=['gene_lower'], errors='ignore')

for col in ['AlphaMissense', 'AlphaMissense_pathogenicity', 'franklin']:
    if col not in df.columns:
        df[col] = None

# === Normalize annotations ===
AM_PATHOGENIC = 0.564
AM_BENIGN = 0.340


def classify_am(val):
    if pd.isna(val): return val
    s = str(val).strip()
    if s in ('likely_pathogenic', 'likely_benign', 'ambiguous'): return s
    try:
        score = float(s)
        if score >= AM_PATHOGENIC:
            return 'likely_pathogenic'
        elif score < AM_BENIGN:
            return 'likely_benign'
        else:
            return 'ambiguous'
    except ValueError:
        return s


def norm_franklin(v):
    if pd.isna(v): return v
    s = str(v).strip().lower()
    mapping = {
        'benign': 'Benign', 'likely benign': 'Likely Benign',
        'vus (low)': 'VUS (low)', 'vus (mid)': 'VUS (mid)',
        'vus mid)': 'VUS (mid)', 'vus(mid)': 'VUS (mid)',
        'vus (high)': 'VUS (high)',
        'pathogenic': 'Pathogenic', 'likely pathogenic': 'Likely Pathogenic',
    }
    return mapping.get(s, v)


if 'AlphaMissense' in df.columns:
    if 'AlphaMissense_raw' not in df.columns:
        df['AlphaMissense_raw'] = df['AlphaMissense']
    df['AlphaMissense'] = df['AlphaMissense'].apply(classify_am)
    print("AlphaMissense normalized:", df['AlphaMissense'].value_counts().to_dict())

if 'franklin' in df.columns:
    if 'franklin_raw' not in df.columns:
        df['franklin_raw'] = df['franklin']
    df['franklin'] = df['franklin'].apply(norm_franklin)
    print("Franklin normalized:", df['franklin'].value_counts().head(5).to_dict())

# Refresh AM/Franklin evaluability now that annotations are merged
df['am_evaluable'] = df['AlphaMissense'].notna()
df['franklin_evaluable'] = df['franklin'].notna()
print(f"AM evaluable: {df['am_evaluable'].sum()}/{len(df)}, "
      f"Franklin evaluable: {df['franklin_evaluable'].sum()}/{len(df)}")


# Recompute external evidence flag (Cell 9 couldn't check AM/Franklin before merge)
def recompute_ext_flag(row):
    if row.get('v6_mechanism') != 'Structure unevaluable':
        return False
    am = ss(row.get('AlphaMissense')).lower()
    fr = std_franklin(row.get('franklin')).lower()
    return (am == 'likely_pathogenic') or (fr in ('pathogenic', 'likely pathogenic', 'vus (high)'))


df['v6_external_evidence_flag'] = df.apply(recompute_ext_flag, axis=1)
n_ext = df['v6_external_evidence_flag'].sum()
if n_ext > 0:
    print(f"External evidence flag: {n_ext} unevaluable variants with AM/Franklin pathogenic support")


# =============================================================================
# FOUR-WAY CONCORDANCE (v6.0)
# =============================================================================

def compute_concordance_v6(row):
    """Four-way concordance: strict, relaxed, T3-inclusive with adjusted denominators."""
    tier = ss(row.get('v6_tier'))
    am = ss(row.get('AlphaMissense')).lower()
    fr = std_franklin(row.get('franklin'))
    fr_lower = fr.lower() if isinstance(fr, str) else ''
    ddg_conf = ss(row.get('ddg_confidence')).lower()

    # A3: Three-axis confidence-gated max_abs_ddg
    # Includes |ddg_monomer| (if monomer_plddt confident),
    # |ddg_fold_partner| and |ddg_binding_partner| (if partner confident).
    ddg_vals = []
    if pd.notna(row.get('ddg_monomer')) and row.get('ddg_monomer_confident', False):
        ddg_vals.append(abs(float(row['ddg_monomer'])))

    _has_pp = False
    for _pl in ALL_PARTNER_LABELS:
        _bind_col = f"ddg_binding_{_pl}"
        _fold_col = f"ddg_fold_{_pl}"
        _conf = row.get(f"ddg_{_pl}_confident", False)
        if _bind_col in row.index and pd.notna(row.get(_bind_col)):
            _has_pp = True
            if _conf:
                ddg_vals.append(abs(float(row[_bind_col])))
        if _fold_col in row.index and pd.notna(row.get(_fold_col)):
            _has_pp = True
            if _conf:
                ddg_vals.append(abs(float(row[_fold_col])))

    if not _has_pp:
        for col in ['ddg_multimer_max', 'ddg_multimer_min']:
            v = row.get(col)
            if pd.notna(v): ddg_vals.append(abs(float(v)))

    max_abs_ddg = max(ddg_vals) if ddg_vals else 0.0

    struct_eval = row.get('structure_evaluable', False)
    ddg_eval = row.get('ddg_evaluable', False)
    am_eval = row.get('am_evaluable', False)
    fr_eval = row.get('franklin_evaluable', False)

    tier_t12 = tier in ('Tier 1', 'Tier 2')
    tier_t123 = tier_t12 or tier == 'Tier 3'

    ddg_strict = 1 if (ddg_conf == 'high' and max_abs_ddg >= DDG_HIGHLY) else 0
    ddg_relaxed = 1 if (ddg_conf not in ('low', '') and max_abs_ddg >= DDG_DESTAB) else 0
    am_strict = 1 if am == 'likely_pathogenic' else 0
    am_relaxed = 1 if am in ('likely_pathogenic', 'ambiguous') else 0
    fr_strict = 1 if fr_lower in ('pathogenic', 'likely pathogenic', 'vus (high)') else 0
    fr_relaxed = 1 if fr_lower in ('pathogenic', 'likely pathogenic', 'vus (high)', 'vus (mid)') else 0

    def build(tier_v, ddg_v, am_v, fr_v):
        s, d = 0, 0
        if struct_eval: s += tier_v; d += 1
        if ddg_eval: s += ddg_v; d += 1
        if am_eval: s += am_v; d += 1
        if fr_eval: s += fr_v; d += 1
        return s, max(d, 1)

    s_s, s_d = build(1 if tier_t12 else 0, ddg_strict, am_strict, fr_strict)
    r_s, r_d = build(1 if tier_t12 else 0, ddg_relaxed, am_relaxed, fr_relaxed)
    t3_s, t3_d = build(1 if tier_t123 else 0, ddg_strict, am_strict, fr_strict)

    return pd.Series({
        'four_way_strict': s_s, 'four_way_strict_denom': s_d,
        'concordance_strict': f"{s_s}/{s_d}",
        'four_way_relaxed': r_s, 'four_way_relaxed_denom': r_d,
        'concordance_relaxed': f"{r_s}/{r_d}",
        'four_way_t3': t3_s, 'four_way_t3_denom': t3_d,
        'concordance_t3': f"{t3_s}/{t3_d}",
        'structure_vote_strict': 1 if tier_t12 else 0,
        'structure_vote_t3': 1 if tier_t123 else 0,
        'ddg_vote_strict': ddg_strict, 'ddg_vote_relaxed': ddg_relaxed,
        'am_vote_strict': am_strict, 'am_vote_relaxed': am_relaxed,
        'franklin_vote_strict': fr_strict, 'franklin_vote_relaxed': fr_relaxed,
        'max_abs_ddg': max_abs_ddg,
    })


print("\nComputing v6.0 concordance...")
conc = df.apply(compute_concordance_v6, axis=1)
for c in conc.columns:
    df[c] = conc[c]

print(f"✓ Concordance: strict 4/4={int((df['four_way_strict'] == 4).sum())}, "
      f"3+/N={int((df['four_way_strict'] >= 3).sum())}, "
      f"relaxed 4/4={int((df['four_way_relaxed'] == 4).sum())}")


# =============================================================================
# PIPELINE 2: NEIGHBORHOOD ±3 (v6.0)
# =============================================================================

def compute_score_nbhd(row):
    """Pipeline 2 score using neighborhood ±3 contacts."""
    score, ev = 0.0, []

    nbhd_eval = row.get('nbhd_mono_evaluable', False)
    if not nbhd_eval or str(nbhd_eval).lower() in ('false', '0', 'nan', ''):
        return pd.Series({
            'nbhd_final_score': np.nan, 'nbhd_score_evidence': 'not_evaluable',
            'nbhd_contact_disruption': np.nan, 'nbhd_total_contacts': np.nan,
            'nbhd_max_inter_weighted': np.nan, 'nbhd_best_inter_partner': '',
            'nbhd_interface_detected': False,
        })

    sev = sf(row.get('substitution_severity'), 0)
    mono_nbhd = sf(row.get('nbhd_mono_contacts_weighted'), 0)

    max_inter_nbhd = 0.0
    best_inter_partner = None
    gated_iface_partners_nbhd = []

    for pl in ALL_PARTNER_LABELS:
        plddt_col = f"multi_{pl}_plddt"
        nbhd_inter_col = f"multi_{pl}_nbhd_inter_weighted"
        nbhd_iface_col = f"multi_{pl}_nbhd_has_interface"

        pl_plddt = sf(row.get(plddt_col), 0) if plddt_col in row.index else 0
        if pl_plddt < 50:
            continue

        if nbhd_iface_col in row.index:
            val = row.get(nbhd_iface_col)
            if val is True or str(val).lower() == 'true':
                gated_iface_partners_nbhd.append(pl)

        if nbhd_inter_col in row.index and pd.notna(row[nbhd_inter_col]):
            v = float(row[nbhd_inter_col])
            if v > max_inter_nbhd:
                max_inter_nbhd = v
                best_inter_partner = pl

    total_nbhd = mono_nbhd + max_inter_nbhd
    disruption = round(sev * total_nbhd, 2)

    for thresh, pts in DISRUPTION_POINTS:
        if disruption >= thresh:
            score += pts;
            ev.append(f'nbhd_disruption({disruption:.1f})')
            break
    else:
        ev.append(f'nbhd_no_disruption({disruption:.1f})')

    n_gated = len(gated_iface_partners_nbhd)
    if n_gated >= 2:
        score += 2.0;
        ev.append(f'nbhd_multi_interface({n_gated})')
    elif n_gated == 1:
        score += 1.5;
        ev.append(f'nbhd_interface({gated_iface_partners_nbhd[0]})')

    # Burial + pLDDT same as Pipeline 1 (single-residue data)
    mono_burial = ss(row.get('monomer_burial'))
    best_rank = BURIAL_RANK.get(mono_burial, 0)
    for pl in ALL_PARTNER_LABELS:
        burial_col = f"multi_{pl}_burial"
        plddt_col = f"multi_{pl}_plddt"
        if burial_col in row.index and pd.notna(row.get(burial_col)):
            if sf(row.get(plddt_col), 0) >= 50:
                pl_rank = BURIAL_RANK.get(ss(row[burial_col]), 0)
                if pl_rank > best_rank:
                    best_rank = pl_rank
    best_burial = RANK_TO_BURIAL.get(best_rank, 'unknown')
    if best_burial == 'buried_core':
        score += 2.0;
        ev.append('buried_core')
    elif best_burial == 'partially_buried':
        score += 1.0;
        ev.append('partially_buried')

    bp = row.get('best_plddt')
    if bp is not None and not pd.isna(bp):
        bp_val = float(bp)
        if bp_val < 50:
            score *= 0.4; ev.append(f'very_low_plddt({int(bp_val)})')
        elif bp_val < 70:
            score *= 0.7; ev.append(f'low_plddt({int(bp_val)})')

    return pd.Series({
        'nbhd_final_score': round(score, 2), 'nbhd_score_evidence': ';'.join(ev),
        'nbhd_contact_disruption': disruption, 'nbhd_total_contacts': total_nbhd,
        'nbhd_max_inter_weighted': max_inter_nbhd,
        'nbhd_best_inter_partner': best_inter_partner or '',
        'nbhd_interface_detected': len(gated_iface_partners_nbhd) > 0,
    })


print("\nComputing Pipeline 2 (neighborhood ±3)...")
if 'nbhd_mono_contacts_weighted' in df.columns:
    nbhd_scores = df.apply(compute_score_nbhd, axis=1)
    for c in nbhd_scores.columns:
        df[c] = nbhd_scores[c]

    df['nbhd_tier'] = df['nbhd_final_score'].apply(assign_tier)
    df['nbhd_structure_evaluable'] = df['nbhd_final_score'].notna()


    # Mechanism (nbhd tier + single-residue DDG/interface)
    def classify_mechanism_nbhd(row):
        if pd.isna(row.get('nbhd_final_score')):
            return 'Structure unevaluable', '', False
        row_copy = row.copy()
        row_copy['v6_tier'] = row.get('nbhd_tier', 'Tier 4')
        row_copy['structure_evaluable'] = True
        return classify_mechanism_v6(row_copy)


    nbhd_mech = df.apply(classify_mechanism_nbhd, axis=1)
    df['nbhd_mechanism'] = [r[0] for r in nbhd_mech]
    df['nbhd_mechanism_partner'] = [r[1] for r in nbhd_mech]


    # Concordance (mirrors Pipeline 1 but uses nbhd_tier)
    def concordance_nbhd(row):
        tier = ss(row.get('nbhd_tier'))
        am = ss(row.get('AlphaMissense')).lower()
        fr = std_franklin(row.get('franklin'))
        fr_lower = fr.lower() if isinstance(fr, str) else ''
        ddg_conf = ss(row.get('ddg_confidence')).lower()
        # A3: Three-axis confidence-gated max_abs_ddg
        ddg_vals = []
        if pd.notna(row.get('ddg_monomer')) and row.get('ddg_monomer_confident', False):
            ddg_vals.append(abs(float(row['ddg_monomer'])))
        _has_pp = False
        for _pl in ALL_PARTNER_LABELS:
            _bind_col = f"ddg_binding_{_pl}"
            _fold_col = f"ddg_fold_{_pl}"
            _conf = row.get(f"ddg_{_pl}_confident", False)
            if _bind_col in row.index and pd.notna(row.get(_bind_col)):
                _has_pp = True
                if _conf:
                    ddg_vals.append(abs(float(row[_bind_col])))
            if _fold_col in row.index and pd.notna(row.get(_fold_col)):
                _has_pp = True
                if _conf:
                    ddg_vals.append(abs(float(row[_fold_col])))
        if not _has_pp:
            for col in ['ddg_multimer_max', 'ddg_multimer_min']:
                v = row.get(col)
                if pd.notna(v): ddg_vals.append(abs(float(v)))
        max_abs_ddg = max(ddg_vals) if ddg_vals else 0.0

        nbhd_eval = row.get('nbhd_structure_evaluable', False)
        ddg_eval = row.get('ddg_evaluable', False)
        am_eval = row.get('am_evaluable', False)
        fr_eval = row.get('franklin_evaluable', False)

        tier_t12 = tier in ('Tier 1', 'Tier 2')
        ddg_strict = 1 if (ddg_conf == 'high' and max_abs_ddg >= DDG_HIGHLY) else 0
        ddg_relaxed = 1 if (ddg_conf not in ('low', '') and max_abs_ddg >= DDG_DESTAB) else 0
        am_strict = 1 if am == 'likely_pathogenic' else 0
        am_relaxed = 1 if am in ('likely_pathogenic', 'ambiguous') else 0
        fr_strict = 1 if fr_lower in ('pathogenic', 'likely pathogenic', 'vus (high)') else 0
        fr_relaxed = 1 if fr_lower in ('pathogenic', 'likely pathogenic', 'vus (high)', 'vus (mid)') else 0

        def build(tv, dv, av, fv):
            s, d = 0, 0
            if nbhd_eval: s += tv; d += 1
            if ddg_eval: s += dv; d += 1
            if am_eval: s += av; d += 1
            if fr_eval: s += fv; d += 1
            return s, max(d, 1)

        s_s, s_d = build(1 if tier_t12 else 0, ddg_strict, am_strict, fr_strict)
        r_s, r_d = build(1 if tier_t12 else 0, ddg_relaxed, am_relaxed, fr_relaxed)
        return pd.Series({
            'nbhd_concordance_strict': f"{s_s}/{s_d}",
            'nbhd_concordance_relaxed': f"{r_s}/{r_d}",
            'nbhd_four_way_strict': s_s, 'nbhd_four_way_relaxed': r_s,
        })


    nbhd_conc = df.apply(concordance_nbhd, axis=1)
    for c in nbhd_conc.columns:
        df[c] = nbhd_conc[c]

    eval_scores = df.loc[df['nbhd_final_score'].notna(), 'nbhd_final_score']
    print(f"✓ Pipeline 2: {len(eval_scores)} evaluable, score range "
          f"{eval_scores.min():.1f}-{eval_scores.max():.1f}")
    print(f"  Neighborhood tier distribution: {df['nbhd_tier'].value_counts().to_dict()}")
else:
    print("  ⚠ No neighborhood data — Pipeline 2 stubbed")
    for col in ['nbhd_tier', 'nbhd_mechanism', 'nbhd_concordance_strict']:
        df[col] = 'Awaiting Phase 1 data'
    df['nbhd_structure_evaluable'] = False


# =============================================================================
# PIPELINE AGREEMENT
# =============================================================================

def classify_agreement(row):
    t1 = ss(row.get('v6_tier'))
    t2 = ss(row.get('nbhd_tier'))
    if 'Awaiting' in t2 or not t2:
        return 'Partially unevaluable'
    t1_high = t1 in ('Tier 1', 'Tier 2')
    t2_high = t2 in ('Tier 1', 'Tier 2')
    if t1_high and t2_high:
        return 'Concordant high'
    elif not t1_high and not t2_high:
        return 'Concordant low'
    elif t2_high and not t1_high:
        return 'Neighborhood-elevated'
    elif t1_high and not t2_high:
        return 'Neighborhood-depressed'
    return 'Partially unevaluable'


df['pipeline_agreement'] = df.apply(classify_agreement, axis=1)
print(f"\n✓ Pipeline agreement: {df['pipeline_agreement'].value_counts().to_dict()}")


# =============================================================================
# VARIANT SUMMARY + gnomAD STUBS
# =============================================================================

def build_variant_summary(row):
    tier = ss(row.get('v6_tier'))
    mech = ss(row.get('v6_mechanism'))
    partner = ss(row.get('v6_mechanism_partner'))
    conc = ss(row.get('concordance_strict'))
    ext = row.get('v6_external_evidence_flag', False)
    mech_display = f"{mech} [{partner}]" if partner else mech
    if ext: mech_display += ' + External evidence'
    return f"{tier} | {mech_display} | Concordance {conc}"


df['variant_summary'] = df.apply(build_variant_summary, axis=1)

for col in ['gnomad_af', 'gnomad_popmax_af', 'gnomad_homozygotes', 'gene_pli', 'gene_loeuf']:
    if col not in df.columns:
        df[col] = np.nan

print(f"\n✓ Annotations complete. gnomAD columns added (awaiting data).")
print(f"\nTop variant summaries (Tier 1/2):")
for _, r in df[df['v6_tier'].isin(['Tier 1', 'Tier 2'])].head(10).iterrows():
    print(f"  {r['gene']} {r['ref_aa']}{int(r['position'])}{r['alt_aa']}: {r['variant_summary']}")


## Cell 9: Save Results

In [ ]:
# =============================================================================
# CELL 11: SAVE RESULTS (v6.0-A3)
# =============================================================================

id_cols = ['gene', 'position', 'ref_aa', 'alt_aa']
grantham_cols = ['grantham_distance', 'grantham_class', 'substitution_severity', 'property_changes']

v6_score_cols = [
    'v6_final_score', 'v6_tier', 'v6_contact_disruption', 'v6_total_contacts',
    'v6_max_inter_contacts', 'v6_best_inter_partner',
    'v6_best_burial', 'v6_best_burial_source',
    'v6_interface_partners_gated', 'v6_n_interface_partners_gated',
    'v6_score_evidence',
]
mech_cols = ['v6_mechanism', 'v6_mechanism_partner', 'v6_external_evidence_flag']
eval_cols = ['structure_evaluable', 'ddg_evaluable', 'am_evaluable', 'franklin_evaluable', 'multimer_evaluable']
am_cols = ['AlphaMissense', 'AlphaMissense_pathogenicity', 'AlphaMissense_raw']
franklin_cols = ['franklin', 'franklin_raw']
gnomad_cols = ['gnomad_af', 'gnomad_popmax_af', 'gnomad_homozygotes', 'gene_pli', 'gene_loeuf']

# A3: DDG columns (three-axis)
ddg_core_cols = [
    'ddg_monomer', 'ddg_monomer_confident', 'ddg_category', 'ddg_category_raw', 'ddg_confidence',
    'ddg_multimer_max', 'ddg_multimer_min', 'ddg_multimer_mean',
    'ddg_category_multimer', 'ddg_category_multimer_raw',
    'n_complexes_tested', 'partners_tested',
    'ddg_low_confidence_flag',
    'ddg_summary_flags', 'ddg_summary_partners_affected', 'ddg_summary_partners_neutral',
]

# Per-partner DDG (binding + fold + interpretation)
per_partner_cols = []
for pl in ALL_PARTNER_LABELS:
    per_partner_cols.extend([
        f"ddg_binding_{pl}", f"ddg_fold_{pl}",
        f"ddg_binding_category_{pl}", f"ddg_fold_category_{pl}",
        f"ddg_{pl}_confident", f"ddg_interp_{pl}",
    ])

conc_cols = [
    'concordance_strict', 'concordance_relaxed', 'concordance_t3',
    'four_way_strict', 'four_way_strict_denom',
    'four_way_relaxed', 'four_way_relaxed_denom',
    'four_way_t3', 'four_way_t3_denom',
    'structure_vote_strict', 'structure_vote_t3',
    'ddg_vote_strict', 'ddg_vote_relaxed',
    'am_vote_strict', 'am_vote_relaxed',
    'franklin_vote_strict', 'franklin_vote_relaxed', 'max_abs_ddg',
]

nbhd_cols = [c for c in df.columns if c.startswith('nbhd_')]
summary_cols = ['variant_summary', 'pipeline_agreement']
mono_cols = [c for c in df.columns if c.startswith('monomer_')]
multi_summary = [
    'n_multimer_complexes', 'multimer_partners',
    'is_interface_any', 'interface_partners', 'n_interface_partners',
    'multimer_plddt_max', 'multimer_plddt_avg',
    'multimer_contacts_max', 'multimer_contacts_avg',
    'multimer_disruption_max', 'multimer_disruption_avg',
]
multi_cols = [c for c in df.columns if c.startswith('multi_')]

v52_cols = [c for c in ['final_score', 'tier', 'final_mechanism', 'pathogenic_mechanism',
                        'contact_disruption', 'total_contacts', 'inter_contacts_sum',
                        'best_burial', 'best_burial_source', 'confidence', 'best_plddt',
                        'score_evidence', 'integrated_class', 'evidence_summary'] if c in df.columns]

ordered = (
    id_cols + grantham_cols + summary_cols +
    v6_score_cols + mech_cols + eval_cols +
    am_cols + franklin_cols + gnomad_cols +
    ddg_core_cols + per_partner_cols +
    conc_cols + nbhd_cols + mono_cols + multi_summary + multi_cols +
    v52_cols
)
seen = set()
final_ordered = []
for c in ordered:
    if c in df.columns and c not in seen:
        final_ordered.append(c)
        seen.add(c)
for c in df.columns:
    if c not in seen:
        final_ordered.append(c)
        seen.add(c)

df_out = df[final_ordered].copy()

out_csv = RESULTS_DIR / "variant_comprehensive.csv"
df_out.to_csv(out_csv, index=False)
print(f"✓ CSV: {out_csv} ({len(df_out)} rows, {len(df_out.columns)} columns)")

hp = df_out[df_out['v6_tier'].isin(['Tier 1', 'Tier 2'])]
hp.to_csv(RESULTS_DIR / "high_priority_variants.csv", index=False)
print(f"✓ High priority: {len(hp)} variants (Tier 1/2)")

foldx_out = RESULTS_DIR / "foldx_ddg_expanded_results.csv"
ddg_exp_cols = ['gene','position','ref_aa','alt_aa','ddg_monomer','ddg_monomer_confident',
                'ddg_category','ddg_category_multimer',
                'ddg_multimer_max','ddg_multimer_min','ddg_multimer_mean',
                'n_complexes_tested','partners_tested',
                'ddg_summary_flags','ddg_summary_partners_affected',
                'ddg_low_confidence_flag']
df[[c for c in ddg_exp_cols if c in df.columns]].to_csv(foldx_out, index=False)
print(f"✓ FoldX DDG expanded: {foldx_out}")

summary_sheet_cols = (
    id_cols + ['grantham_distance'] +
    ['v6_final_score', 'v6_tier', 'v6_mechanism', 'v6_mechanism_partner'] +
    ['concordance_strict', 'concordance_relaxed', 'concordance_t3'] +
    ['AlphaMissense', 'franklin',
     'ddg_monomer', 'ddg_monomer_confident', 'ddg_category',
     'ddg_category_multimer', 'ddg_low_confidence_flag',
     'ddg_summary_flags', 'ddg_summary_partners_affected'] +
    ['structure_evaluable', 'ddg_evaluable', 'multimer_evaluable'] +
    gnomad_cols +
    ['nbhd_tier', 'nbhd_mechanism', 'nbhd_concordance_strict'] +
    ['pipeline_agreement', 'variant_summary', 'v6_external_evidence_flag']
)

try:
    import openpyxl
    xlsx_path = RESULTS_DIR / "variant_comprehensive.xlsx"
    with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
        df[[c for c in summary_sheet_cols if c in df.columns]].to_excel(writer, sheet_name='Summary', index=False)
        df[id_cols + [c for c in mono_cols if c in df.columns]].to_excel(writer, sheet_name='Monomer Detail', index=False)
        df[id_cols + [c for c in (multi_summary + multi_cols) if c in df.columns]].to_excel(writer, sheet_name='Multimer Detail', index=False)
        ddg_detail = id_cols + [c for c in (ddg_core_cols + per_partner_cols) if c in df.columns]
        df[ddg_detail].to_excel(writer, sheet_name='DDG Detail', index=False)
        conc_detail = id_cols + [c for c in (eval_cols + conc_cols + nbhd_cols) if c in df.columns]
        df[conc_detail].to_excel(writer, sheet_name='Concordance Detail', index=False)
    print(f"✓ XLSX: {xlsx_path} (5 sheets)")
except ImportError:
    print("⚠ openpyxl not installed — XLSX skipped")

print(f"\n{'=' * 70}")
print(f"MAVIS Pipeline — COMPLETE")
print(f"{'=' * 70}")
print(f"  Variants: {len(df_out)}")
print(f"  Columns: {len(df_out.columns)}")
print(f"  Pipeline 1 tiers: {df_out['v6_tier'].value_counts().to_dict()}")
if 'nbhd_tier' in df_out.columns:
    print(f"  Pipeline 2 tiers: {df_out['nbhd_tier'].value_counts().to_dict()}")
print(f"  Pipeline agreement: {df_out['pipeline_agreement'].value_counts().to_dict()}")
print(f"  Strict 4/4: {int((df_out['four_way_strict'] == 4).sum())}")
print(f"  Mechanism categories: {df_out['v6_mechanism'].nunique()}")
print(f"  DDG per-partner pairs: {sum(df_out[f'ddg_binding_{pl}'].notna().sum() for pl in ['actin','actb','dvl2','cdh2_truncated','ctnnb1','cdh2_cyto','actb_no_bind','rock2','gli3','kpna1','kpna6','mdfi','tcf7l1'])}")
print(f"{'=' * 70}")
